In [5]:
# Базовые библиотеки для воспроизводимости, работы с данными и удобного вывода результатов.
import os
import sys
import random
import subprocess
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call(["uv", "pip", "install", "-q", package_name])


# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("FAISS недоступен, будет использован fallback на sklearn NearestNeighbors.")
    print("Причина:", repr(e))


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)


NumPy: 2.4.2
Pandas: 3.0.1
FAISS available: True


In [6]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cuda


# НУЖНА БДшка типа по той что указано в семинаре и можешь тупо скопировать их "best practice" state of art подход

In [22]:
# Небольшой учебный корпус документов по теме retrieval и RAG.
documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_01",
        "title": "Эмбеддинги текстов",
        "text": (
            "Эмбеддинг – это плотное векторное представление текста, в котором семантически похожие фразы "
            "располагаются близко друг к другу. Такие векторы позволяют искать похожие документы не по "
            "точному совпадению слов, а по смыслу запроса. Для задач retrieval эмбеддинги обычно "
            "нормализуют, чтобы косинусное сходство можно было считать через скалярное произведение."
        ),
    },
    {
        "doc_id": "doc_02",
        "title": "FAISS и быстрый поиск",
        "text": (
            "FAISS – библиотека для эффективного поиска ближайших соседей по векторам. "
            "В учебных примерах удобно начинать с точного индекса IndexFlatIP или IndexFlatL2, "
            "а затем переходить к более сложным структурам. Если эмбеддинги нормализованы, "
            "IndexFlatIP хорошо подходит для поиска по косинусному сходству."
        ),
    },
    {
        "doc_id": "doc_03",
        "title": "Чанкинг документов",
        "text": (
            "Перед индексацией длинные документы обычно режут на чанки – небольшие фрагменты текста. "
            "Слишком крупные чанки дают много лишнего контекста, а слишком мелкие могут терять смысл. "
            "На практике часто используют overlap, чтобы важная мысль не разрывалась на границе соседних фрагментов."
        ),
    }
]

docs_df = pd.DataFrame(documents)
print("Размер корпуса:", len(docs_df))
display(docs_df[["doc_id", "title"]])

import wikipediaapi

wiki = wikipediaapi.Wikipedia(user_agent='User-Agent: CoolBot/0.0 (https://example.org/coolbot/; coolbot@example.org) generic-library/0.0', language='ru')
titles = ["Метрополитен", "Тоннель", "Эскалатор", "Московский метрополитен", "Вентиляция"]
corpus = []

for title in titles:
    page = wiki.page(title)
    if page.exists():
        corpus.append({"title": title, "text": page.text})
        display(Markdown(f"## Title: `{page.title}`"))
        display(Markdown(page.text))

print(f"Загружено {len(corpus)} статей")


Размер корпуса: 3


,doc_id,title
0,doc_01,Эмбеддинги текстов
1,doc_02,FAISS и быстрый поиск
2,doc_03,Чанкинг документов


## Title: `Метрополитен`

Метрополите́н (от англ. Metropolitan, сокр. от Metropolitan Railway — от греч. μητρόπολις – глав­ный го­род, сто­ли­ца), ме́тро (фр. métro, англ. metro, сокр. the met, рус. метро́), скоростной транзи́т (англ. rapid transit), подземная железная дорога (англ. underground railway),  или просто «подземка» (англ. subway, сокр. the sub, а также брит. англ. underground либо жарг. tube) — вид рельсового скоростного внеуличного электрического пассажирского транспорта города или агломерации. При сооружении метрополитена в условиях сложившейся городской застройки наибольшее распространение получили подземные линии в тоннелях. Они не стесняют уличное движение и слабо нарушают подземные коммуникации и планировку города. Технические решения имеют множество особенностей в зависимости от конкретной страны, города и периода эксплуатации. Впервые в мире метрополитен открылся в Лондоне 10 января 1863 года.
Может быть подземным (в тоннелях (перегонах)), наземным (на поверхности земли) и надземным (на эстакадах).
Движение поездов в метрополитене регулярное, согласно графику движения. Мировым лидером по нему является Московский. Этому виду транспорта свойственны высокая скорость (до 80 км/ч по АЛС-АРС на метрополитенах России) и провозная способность (до 60 тыс. пассажиров в час в одном направлении).

Особенности метрополитенов и статистика
Крупнейшие метрополитены в мире на ноябрь 2025 года:

по количеству линий и маршрутов — Нью-Йоркский (36 линий, 29 маршрутов, Пекинский и Сеульский);
по длине линий — Шанхайский (743 км) и Пекинский (727 км);
по количеству станций — Пекинский (490);
по средней глубине станций — Петербургский (в среднем от ≈ 40 м);
по годовому пассажиропотоку — Пекинский и Токийский;
по суточному пассажиропотоку — Пекинский.
Самые короткие (короче 9 км) метрополитены: в индийской Патне, венесуэльском Маракайбо, итальянских Катании и Генуе, украинском Днепре.
Лозанна, Брешиа и Ренн — самые маленькие города мира, имеющие метрополитен.
Самой длинной линией в мире является 3-я линия метрополитена Гуанчжоу — 68,53 км. К 2030 году она уступит этот статус линии 15 Парижского метрополитена, длина которой должна будет составить 75 км.
Самой глубокой станцией в мире с 25 января 2022 года является «Хунъяньцунь» Чунцинского метрополитена — 116 м. Ранее, с 6 ноября 1960 года — открытия первой очереди Киевского метрополитена, этот статус принадлежал станции «Арсенальная» глубиной 105,5 м, у которой, однако, в отличие от самой глубокой в России «Адмиралтейской» (86 м) Петербургского метрополитена, глубина обусловлена не заложением станции, а возвышенностью местности.
Станция «Воробьёвы горы» Московского метрополитена — первая в мире, расположенная на метромосту.
Станция «Мякинино» Московского метрополитена — единственная в России, построенная с участием частных инвестиций и не принадлежащая предприятию, а «Тверская» — первая в мире глубокого заложения, построенная на действующем участке без остановки движения поездов.
Советско-российская серия вагонов метрополитена 81-717/714 является мировым лидером по распространённости (эксплуатируется в метрополитенах 18 городов стран Варшавского договора, бывших республик СССР и Соцлагеря) и длительностью производства (с 1976 по 2021 годы).
Поезд «НеВа» Петербургского метрополитена лишён крана машиниста ввиду отсутствия тормозной магистрали, присутствующих или присутствовавших у остальных поездов производства заводов в том числе ЗАО «Вагонмаш», ОЭВРЗ, ММЗ и АО «Метровагонмаш».
На челночном ответвлении Ереванского метрополитена эксплуатируются единственные в мире одиночные двухкабинные вагоны (электромотрисы) модели 81-717М в количестве двух единиц — № 0112—8783.
Перегон Московско-Петроградской линии Петербургского метрополитена «Невский проспект» — «Горьковская» имеет предельно допустимый Правилами технической эксплуатации (ПТЭ) уклон крутизной 60 ‰ (тысячных — спуск на 6 метров через каждые 100 метров), что связано с прохождением тоннеля под впадиной дна подземной реки: при проектировании линии из-за обводнённости грунтов отказались от станции в районе Марсова поля.
Красная линия Дубайского метрополитена является самой длинной в мире, функционирующей в полностью автоматическом режиме (с беспилотными поездами), благодаря чему в 2011 году попала в Книгу рекордов Гиннеса.
Пожар в Бакинском метрополитене 1995 года является крупнейшей трагедией по числу жертв всех метрополитенов мира, вызванной человеческим фактором.
Монреальский метрополитен — крупнейший в мире из полностью подземных.

Определение метрополитена
Большое разнообразие систем внеуличного скоростного городского и пригородного транспорта делает затруднительной их однозначную классификацию. Все определения метрополитена условны. В отношении многих транспортных систем можно с уверенностью сказать, что они являются (или, наоборот, не являются) метрополитенами, но в то же время существует ряд «пограничных» и «гибридных» транспортных систем.
Конечное решение о том, отнести ту или иную транспортную систему к метрополитенам или нет, зависит от принятого определения или может делаться эмпирически. Например, в Зерфаусе и Новом Афоне существуют подземные железные дороги, имеющие некоторое сходство с подземными линиями метрополитена и потому неформально часто так и называемые, однако если обязательным признаком метрополитена считать расположение в городе, то метрополитенами они не являются.

Определяющие признаки метрополитена
Создатель сайта urbanrail.net и автор нескольких книг о метрополитене Роберт Швандль предлагает следующие определяющие признаки метрополитена:

используется в урбанизированной местности (в городах и городских агломерациях);
работает на электротяге;
полностью отделена от любого другого движения;
работает часто (с рабочим интервалом в дневное время не более 30 минут).
Также он предлагает ещё один признак: совпадение уровня пола вагона и перрона, но этот признак не обязателен. При этом не указано, какой должна быть путевая инфраструктура. То есть, по Швандлю, она может быть практически любой: традиционные рельсовые пути (традиционный метрополитен), ALWEG (наиболее распространённый тип городских пассажирских монорельсов) и так далее.
Это определение обладает некоторой условностью. Под него не подпадает, например, Чикагский метрополитен, который имеет несколько одноуровневых пересечений с дорогами (хотя Швандль всё равно рассматривает эту систему как метрополитен в виде исключения). Ряд метрополитенов в прошлом работали на иных видах тяги (паровая, канатная), а в некоторых метрополитенах имеются отдельные участки с низкой интенсивностью работы.
В частности, Швандль рассматривает Вуппертальскую подвесную дорогу как полноценную систему метрополитена, что может быть недопустимо с точки зрения других определений, накладывающих разные по строгости ограничения на техническую реализацию метрополитена.
В законодательстве ЕАЭС метрополитен определяется как

Неопределяющие признаки
Как правило, транспортные специалисты не считают определяющим признаком способ размещения трассы (подземный, наземный, надземный), хотя в России и странах бывшего СССР исторически сложилось представление о метрополитене именно как о подземном виде транспорта.
Также не является определяющей принятая система токосъёма (контактный провод, контактный рельс). Хотя для метрополитена чаще, чем для других видов транспорта, характерно использование контактного рельса, нередко встречаются и иные технические решения.

Основные свойства
В мегаполисах со сложившейся застройкой линии метро, как правило, проложены под землёй и лишь иногда выходят на поверхность или на эстакады. Габариты и масса подвижного состава могут достигать железнодорожных стандартов, хотя обычно уступают им. Метропоезда насчитывают, как правило, 4—8 вагонов. Диаметр тоннелей достигает 4—7 метров (но во многих системах встречаются и более узкие тоннели, например в Берлине ширина узкопрофильных тоннелей — всего 2,3 метра), предельные уклоны больше, чем на железных дорогах общего назначения, но меньше, чем на трамвае, минимальные радиусы закругления значительно больше трамвайных. Платформы на станциях обычно имеют длину 100—165 м и ширину 5—20 м. Линии метрополитена обычно проходят вдоль градообразующих осей и являются каркасом городской пассажирской транспортной системы. Стоимость сооружения метрополитена сильно зависит от условий окружающей среды и применяемых технологий строительства. Типичная стоимость километра подземной линии мелкого заложения — порядка 30 млн. долл. США (без учёта стоимости строительства станций).
В разных странах исполнение и параметры метрополитенов могут варьироваться (например, существуют почти полностью наземные системы), но отличительными чертами метрополитена являются: использование электрической тяги, высокая интенсивность и скорость движения поездов и большой пассажиропоток и, естественно, полная обособленность от прочего городского транспортного движения.
Размеры метрополитенов находятся в диапазоне от 2-километровой линии «мини-метро» в израильской Хайфе (см. Кармелит) до Нью-Йоркской системы «подземок» и «надземок» с общей протяжённостью линий более 1300 км.
Разновидностями метрополитена или близкими к нему по свойствам и назначению транспортными системами (в зависимости от принятого определения) являются лёгкое метро, преметро, S-Bahn (S-Tog и т. п.), городские монорельсы (кроме аттракционов и экскурсионных).

Этимология
Название «метрополитен» (метро) принято во многих странах. Подавляющее большинство действующих метрополитенов мира представляет собой разновидность железной дороги, однако по устройству инфраструктуры и подвижного состава различимы с ней.
Первая подземная железная дорога была построена в Лондоне в 1863 году компанией Metropolitan Railway — буквально «столичная железная дорога». Однако в английском языке название этой линии нарицательного характера не приобрело. Появлявшиеся вслед за тем линии метрополитена, в том числе и в Великобритании, получали другие названия. В частности, первой в мире линией является Метропо́литен Лондонского метро, само же метро в английском языке называется «Лондон-андеграунд» (англ. London Underground, «лондонская подземная железная дорога» или, также, «лондонская подземка» или — в разговорной речи — «тьюб» (англ. tube, «труба»).
Нарицательный смысл слово «метрополитен» и общепринятое сокращение «метро» приобрели в Париже. Чтобы город не оказался зависимым от национальной администрации железных дорог, при строительстве парижского метрополитена было решено создать отдельную компанию, которая получила название Парижской компании столичной железной дороги (фр. Compagnie du chemin de fer métropolitain de Paris; слово фр. métropolitain («метрополитен») во французском языке носило нарицательное значение «столичный»). Постепенно слова «метрополитен» и «метро» приобрели значение городской внеуличной железной дороги вообще во французском языке, а потом в таком качестве пришли и в другие языки (в том числе и русский).
Кроме того, Максимом Горьким в книге «Город Жёлтого Дьявола» было введено в русский язык слово-калька «подземка». Оно прижилось, но преимущественно в качестве обозначения зарубежных метрополитенов (лондонская подземка, нью-йоркская подземка и т. д.), хотя в последнее время встречается в российской прессе и применительно к российским метрополитенам, проложенным в основном под землёй. Соответственно, преимущественно эстакадные метрополитены называют «надземками».
Своя система связанной с метрополитеном терминологии используется в немецкоязычных странах. В настоящее время наиболее распространённые термины — U-Bahn и S-Bahn. Термин U-Bahn является сокращением от нем. Untergrundbahn — букв. «подземная железная дорога». U-Bahn близок к метро в традиционном российском понимании, так как является внутригородским транспортом, в основном расположенным под землёй. В некоторых городах (Кёльн, Дюссельдорф) слово U-Bahn используется для обозначения подземных участков трамвайных линий. S-Bahn (от нем. Stadtbahn — городская железная дорога. В Берлине первоначально называлась SS-Bahn от нем. Schnellstadtbahn — скоростная городская железная дорога). S-Bahn ближе к пригородным железнодорожным поездам. В городах S-Bahn иногда имеет подземные участки (U-Bahn). В настоящее время термин S-Bahn обычно не расшифровывается как сокращение и означает городские (пригородно-городские) поезда, а термин нем. Stadtbahn принял другое значение — городская железная дорога как легкорельсовый транспорт. Также имеется термин Hochbahn, означающий трассы метро, проложенные на эстакадах — «надземки».
В английском языке в нарицательном смысле применяется термин англ. rapid transit (скоростной городской транспорт), однако употребляется он только тогда, когда по смыслу невозможно ограничиться названием одной конкретной системы метрополитена. В остальных случаях используются индивидуальные названия: в Лондоне — англ. London Underground, в Нью-Йорке — англ. New York Subway, в Ливерпуле — англ. Merseyrail, в Вашингтоне — англ. Washington Metrorail, в Сан-Франциско — англ. BART и т. п. В некоторых городах применяется название «метро» (англ. metro) для систем, по своему характеру близких к метро, или для всего городского транспорта (собственно метро и наземный пассажирский транспорт (в том числе автобусы и трамваи)) в совокупности.

История
Первая линия метрополитена длиной 6 км была построена в Лондоне. Запущена 10 января 1863 года. Изначально первая линия в Лондоне эксплуатировалась на паровой тяге, которая начиная с 1890 года заменялась на электрическую.
Второй метрополитен был открыт в Нью-Йорке в 1868 как надземный, однако первые надземные участки не сохранились и впоследствии были заменены подземными (первая подземная линия открыта в 1904).
6 июня 1892 — открыта первая надземная линия метрополитена Чикаго на паровой тяге.
В Европе старейшими после Лондонского метрополитена являются метрополитены Будапешта (1896), Глазго (1896), Парижа (1900), Берлина (1902), Гамбурга (1912).
Иногда к числу старейших метрополитенов Европы причисляют стамбульский «Тюнель» (европейская часть города, 1875), несмотря на то что он является, по сути, подземным фуникулёром (полноценный Стамбульский метрополитен открылся только в 2000 году), и Афинский метрополитен, который, однако же, в момент открытия (1869) представлял собой обычный городской поезд; в 1904 году линия была электрифицирована с использованием третьего рельса, с этого момента её хоть как-то можно причислять к метрополитенам. Также не относится к числу старейших и Венский метрополитен: в 1898 году в Вене открылась городская железная дорога, а в 1966 году — подземный трамвай, лишь в 1976 году ставший основой полноценного метрополитена.
В Советском Союзе первая линия метрополитена была торжественно открыта в Москве 15 мая 1935 года. На территории СССР метрополитен был открыт также в Ленинграде (1955), Киеве (1960), Тбилиси (1966), Баку (1967), Харькове (1975), Ташкенте (1977), Ереване (1981), Волгограде (скоростной трамвай) и Минске (1984), Горьком (1985), Новосибирске и Кривом Роге (скоростной трамвай) (1986), Куйбышеве (1987) и Свердловске (1991).
После распада СССР метрополитен был открыт всего лишь в трёх бывших советских городах: Днепропетровске (1995, Украина), Казани (2005, Россия) и Алма-Ате (2011, Казахстан).

Хронография действующих систем метрополитена
XIX век
1863 — Лондон (Великобритания).
1868 — Нью-Йорк (США) — открылся как надземная железная дорога. Первая подземная линия открылась в 1904 году. Три системы различных операторов объединены в одну в 1940 году.
1869 — Афины (Греция) — открылся как городская железная дорога. Переведён на электрическую тягу в 1904 году. Железная дорога преобразована в метрополитен в 2000 году.
1875 — Стамбул (Османская империя), подземный фуникулёр.
1892 — Чикаго (США) — открылся как надземная железная дорога. Первая подземная линия открылась в 1943 году.
1893 — Ливерпуль (Великобритания), надземная железная дорога. Закрыта в 1956 году.
1896 — Будапешт (Австро-Венгерская империя) — первый метрополитен на Европейском континенте, первый метрополитен в мире на электрической тяге; Глазго (Великобритания).
1897 — Бостон (США).
1900 — Париж (Франция).

XX век
1901 — Вупперталь (Германия), подвесная дорога
1902 — Берлин (Германия)
1907 — Филадельфия (США)
1908 — PATH (США) — подземная железная дорога, связавшая тоннелем под рекой Гудзоном Нью-Йорк и территорию штата Нью-Джерси; Генуя (Италия) — открылся трамвайный тоннель, с 1990 года входит в состав метрополитена.
1912 — Гамбург (Германия)
1913 — Буэнос-Айрес (Аргентина)
1918 — Сан-Франциско (США) — открылся трамвайный тоннель с двумя подземными станциями; с 1980 года входит в состав лёгкого метро.
1919 — Мадрид (Испания)
1924 — Барселона (Испания)
1927 — Токио (Япония)
1928 — Осло (Норвегия) — сеть пригородных трамваев с подземными участками; переоборудована в метрополитен в 1966 году; Рочестер (США), подземный трамвай, закрытый в 1956 году.
1933 — Осака (Япония), Стокгольм (Швеция), трамвайный тоннель; с 1950 года входит в состав метрополитена.
1935 — Москва (СССР), Ньюарк (США), лёгкое метро
1950 — Стокгольм (Швеция)
1954 — Торонто (Канада)
1955 — Кливленд (США), Ленинград (СССР; ныне — Санкт-Петербург), Рим (Италия)
1957 — Брюссель (Бельгия), трамвайный тоннель, Нагоя (Япония)
1959 — Лиссабон (Португалия), Хайфа (Израиль), подземный фуникулёр
1960 — Киев (СССР)
1964 — Милан (Италия)
1966 — Вена (Австрия), трамвайный тоннель (с 1980 года частично входит в состав метрополитена), Монреаль (Канада), Осло (Норвегия), Тбилиси (СССР)
1967 — Баку (СССР), Эссен (Германия), скоростной трамвай
1968 — Кёльн (Германия), скоростной трамвай, Роттердам (Нидерланды), Франкфурт-на-Майне (Германия)
1969 — Мехико (Мексика), Пекин (Китай)
1971 — Мюнхен (Германия), Саппоро (Япония)
1972 — Иокогама (Япония), Нюрнберг (Германия), Сан-Франциско (США), метроэлектричка
1973 — Пхеньян (КНДР)
1974 — Прага (Чехословакия), Сан-Паулу (Бразилия), Сеул (Южная Корея)
1975 — Антверпен (Бельгия), трамвай, пре-метро, Бонн (ФРГ), лёгкое метро, Ганновер (ФРГ), лёгкое метро, Новый Афон (СССР), миниметро, Сантьяго (Чили), Харьков (СССР)
1976 — Брюссель (Бельгия), Вашингтон (США), Вена (Австрия), Шарлеруа (Бельгия), лёгкое метро
1977 — Амстердам (Нидерланды), Кобе (Япония), Марсель (Франция), Мюльхайм-на-Руре (Германия), скоростной трамвай, Ташкент (СССР), Ливерпуль (Великобритания), метроэлектричка
1978 — Лион (Франция), Эдмонтон (Канада), лёгкое метро
1979 — Атланта (США), Бохум (Германия), скоростной трамвай, Бухарест (Румыния), Гонконг (Британская империя), Рио-де-Жанейро (Бразилия)
1980 — Ньюкасл-апон-Тайн (Великобритания), Сан-Франциско (США), миниметро
1981 — Дюссельдорф (Германия), скоростной трамвай, Ереван (СССР), Калгари (Канада), Киото (Япония), Фукуока (Япония)
1982 — Хельсинки (Финляндия)
1983 — Балтимор (США), Каракас (Венесуэла), Лилль (Франция)
1984 — Волгоград (СССР), подземный трамвай, Гельзенкирхен (Германия), скоростной трамвай, Дортмунд (Германия), скоростной трамвай, Калькутта (Индия), Майами (США), надземка, Манила (Филиппины), Минск (СССР), Питтсбург (США), лёгкое метро, Тяньцзинь (Китай)
1985 — Буффало (США), лёгкое метро, Горький (СССР; ныне — Нижний Новгород), Зерфаус (Австрия), подземная дорога на воздушной подушке, Китакюсю (Япония), монорельс, Порту-Алегри (Бразилия), надземка, Пусан (Южная Корея), Ресифи (Бразилия), Штутгарт (Германия), скоростной трамвай
1986 — Белу-Оризонти (Бразилия), Ванкувер (Канада), Кривой Рог (СССР), скоростной трамвай, Новосибирск (СССР), Цюрих (Швейцария), трамвайный тоннель
1987 — Детройт (США), монорельс, Каир (Египет), Куйбышев (СССР; ныне — Самара), Лондон, лёгкое метро, Сингапур (Сингапур), Сэндай (Япония)
1988 — Валенсия (Испания)
1989 — Гвадалахара (Мексика), Джексонвилл (США), Херне (Германия), скоростной трамвай
1990 — Генуя (Италия)
1991 — Билефельд (Германия), метротрам, Лозанна (Швейцария), Монтеррей (Мексика), Свердловск (СССР; ныне — Екатеринбург), Терезина (Бразилия), лёгкое метро
1992 — Дуйсбург (Германия), метротрам, Саас-Фе (Швейцария), подземный фуникулёр, Тулуза (Франция)
1993 — Лос-Анджелес (США), Неаполь (Италия), Сент-Луис (США), лёгкое метро, Шанхай (Китай)
1994 — Хиросима (Япония), Руан (Франция), скоростной трамвай
1995 — Бильбао (Испания), Варшава (Польша), Днепропетровск (Украина; ныне — Днепр), Куала-Лумпур (Малайзия), надземное метро, Медельин (Колумбия), надземка, Ченнаи (Индия), надземное метро
1996 — Анкара (Турция), Тайбэй (Тайвань)
1997 — Гуанчжоу (Китай), Тэгу (Южная Корея)
1998 — София (Болгария)
1999 — Бангкок (Таиланд), надземное метро, Инчхон (Южная Корея), Катания (Италия), Солт-Лейк-Сити (США), лёгкое метро, Тегеран (Иран)

XXI век
00-е
2000 — Измир (Турция), Стамбул (Турция), Стокгольм (скоростной трамвай Tvärbanan с подземными участками)
2001 — Бразилиа (Бразилия), Лима (Перу) (тестовая эксплуатация), Оттава (Канада) (лёгкое метро)
2002 — Бурса (Турция), Дели (Индия), Копенгаген (Дания), Рен (Франция), Чанчунь (Китай) (лёгкое метро)
2003 — Далянь (Китай), Лима (Перу) (регулярная эксплуатация), Наха (Япония) (монорельс)
2004 — Гаага (Нидерланды) (трамвайный тоннель), Кванджу (Южная Корея), Лас-Вегас (США) (монорельс), Миннеаполис (США) (метротрам), Москва (Россия) (монорельс) (закрыт в 2025 году), Сан-Кристина (Италия) (подземный фуникулёр), Сан-Хуан (Пуэрто-Рико), Ухань (Китай), Шэньчжэнь (Китай)
2005 — Вальпараисо (Чили), Казань (Россия), Нанкин (Китай), Порту (Португалия), метротрам, Чунцин (Китай) (надземное метро)
2006 — Валенсия (Венесуэла), Маракайбо (Венесуэла), Турин (Италия), Тэджон (Корея)
2007 — Аликанте (Испания) (метротрам), Пальма-де-Мальорка (Испания)
2008 — Гаосюн (Тайвань), Краков (Польша) (трамвайный тоннель), Перуджа (Италия) (миниметро)
2009 — Адана (Турция), лёгкое метро, Дубай (ОАЭ), Санто-Доминго (Доминиканская Республика), Севилья (Испания) (лёгкое метро)

10-е
2010 — Чэнду (КНР), Шэньян (КНР), Фошань (КНР), Мекка (Саудовская Аравия), Белград (Сербия), метроэлектричка, Мальмё (Швеция), метроэлектричка, Венеция (Италия), миниметро
2011 — Мешхед (Иран), Бангалор (Индия), Сиань (КНР), Алжир (Алжир), Алма-Ата (Казахстан)
2012 — Сучжоу (КНР), Куньмин (КНР), Ыйджонбу (Южная Корея), надземное метро, Познань (Польша), трамвайный тоннель, Форталеза (Бразилия), Ханчжоу (КНР).
2013 — Брешиа (Италия), Харбин (КНР), Гургаон (Индия), надземное метро, Чжэнчжоу (КНР), Лейпциг (Германия), метроэлектричка
2014 — Панама (Панама), Чанша (КНР), Нинбо (КНР), Мумбаи (Индия), надземное метро, Салвадор (Бразилия), Уси (КНР), Малага (Испания), метротам, Шираз (Иран).
2015 — Джайпур (Индия), Ченнаи (Индия), Тебриз (Иран), Аддис-Абеба (Эфиопия), лёгкое метро, Исфахан (Иран), Циндао (КНР), Наньчан (КНР)
2016 — Фучжоу (КНР), Дунгуань (КНР), Наньнин (КНР), Хэфэй (КНР), Ашхабад (Туркменистан), монорельс
2017 — Таоюань (Тайвань), Коччи (Индия), надземное метро, Шицзячжуан (КНР), Чанчунь (КНР), Лакхнау (Индия), Гранада (Испания), скоростной трамвай, Хайдарабад (Индия), надземное метро, Гуйян (КНР), Сямынь (КНР).
2018 — Абуджа (Нигерия), ЛРТ, Урумчи (КНР).
2019 — Цзинань (КНР), Вэньчжоу (КНР), Ноида (Индия), надземное метро, Ахмадабад (Индия), надземное метро, Нагпур (Индия), надземное метро, Джакарта (Индонезия), Доха (Катар), Сидней (Австралия), Ланьчжоу (КНР), Чанчжоу (КНР), Сюйчжоу (КНР), Аомынь (бывш. Макао), лёгкое метро, Хух-Хото (КНР).

20-е
2020 — Лахор (Пакистан), Тайюань (КНР).
2021 — Лоян (КНР), Тайчжун (Тайвань), Шаосин (КНР), Уху (КНР), Ханой (Вьетнам), Канпур (Индия).
2022 — Пуна (Индия), Цзиньхуа (КНР), Наньтун (КНР), Дакка (Бангладеш).
2023 — Кередж (Иран), Кито (Эквадор), Лагос (Нигерия), надземка, Нави Мумбаи (Индия), надземка, Гонолулу (США), лёгкое метро.
2024 — Агра (Индия), Салоники (Греция), Эр-Рияд (Саудовская Аравия), Хошимин (Вьетнам).
2025 — Индаур (Индия), Патна (Индия), Бхопал (Индия).

Строящиеся и планируемые метрополитены
2026 — Астана (Казахстан), лёгкое метро (начались тестовые испытания), Ахваз (Иран) (строится), Керманшах (Иран) (строится), Бэнбу (КНР), монорельс (строится), Кум (Иран) (строится), Мератх (Индия) (строится), Синин (КНР) (планируется), Тривандрам (Индия), ЛРТ (планируется), Гебзе (Турция, строится), Мерсин (Турция) (строится).
2027 — Дублин (Ирландия) (планируется).
2028 — Белград (Сербия) (строится), Богота (Колумбия) (планируется), Абиджан (Кот-д’Ивуар), надземка (строится).
2035 — Копенгаген (Дания) / Мальмё (Швеция) (планируется) — международная линия метро через пролив Эресунн.
без точной даты — Кордова (Аргентина) (планируется), Гояния (Бразилия) (планируется), Куритиба (Бразилия) (планируется), Флорианополис (Бразилия) (планируется), Гуаренас/Гуатирре (Венесуэла), ЛРТ (строится), Варанаси (Индия) (планируется), Виджаявада (Индия), надземка (планируется), Вишакхапатнам (Индия) (планируется), Гувахати (Индия) (планируется), Кожикоде (Индия), ЛРТ (планируется), Лудхиана (Индия) (планируется), Сурат (Индия) (планируется), Болонья (Италия) (планируется), Найроби (Кения), ЛРТ (планируется), Баотоу (КНР) (стройка законсервирована), Ичан (КНР) (планируется), Улан-Батор (Монголия) (планируется), Абу-Даби (ОАЭ) (планируется), Карачи (Пакистан) (планируется), Даммам (Саудовская Аравия) (планируется), Джидда (Саудовская Аравия) (планируется), Медина (Саудовская Аравия) (планируется), Кампала (Уганда), ЛРТ (планируется), Кавасаки (Япония) (планируется), Вильнюс (Литва) (планируется), Душанбе (Таджикистан) (планируется).

Строительство
Строительство метро стоит очень дорого, и поэтому бывает экономически оправдано только в крупных городах (территориально или по численности населения). В СССР таковыми считались города с численностью населения от 1 млн жителей. Различают закрытый способ строительства (с помощью тоннелепроходческих щитов) и открытый, при котором тоннели и станции строятся соответственно в траншеях и котлованах и, будучи завершёнными, снова засыпаются грунтом.
Закрытый способ применяется при строительстве линий глубокого заложения, когда этого требуют гидрогеологические условия или необходимо сохранить ценную застройку в городах. В иных случаях станции мелкого заложения строят открытым способом. Для линий мелкого заложения в России применяют также гибридный — «московский» — способ, когда станции строятся открытым способом, а тоннели — закрытым. Коммуникации необходимо переносить (когда они там есть) лишь в зонах строительства станций, а в зонах прокладки перегонных тоннелей такой необходимости нет. Также не нужно временно закрывать дороги и т. д., поэтому строительство оказывается дешевле. По ценам 2006 года стоимость 1 км тоннеля, построенного открытым способом, составляет приблизительно 1,4 млрд руб., а 1 км тоннеля, построенного закрытым способом, — около 2—2,2 млрд руб. Необходимо также учитывать, что эти цифры приведены для одного однопутного тоннеля. Учитывая, что линии метрополитена, как правило, строится двухпутными и, как это обычно делается в России, каждый линейный путь прокладывается в отдельном тоннеле, — линия метро получается двухтоннельная. Следовательно, при расчётах стоимости строительства километра линии метро стоимость строительства километра однопутного тоннеля следует умножать на 2.

Подвижной состав
Электропоезд метрополитена состоит из нескольких вагонов: двух головных вагонов, имеющих кабины управления и от одного до шести промежуточных вагонов, прицепленных между ними. Вагон метро обычно длиннее трамвайного, но короче железнодорожного. На вагонах российского производства серии 81-717/714 в середине вагона над дверьми находятся три сигнальные лампочки, которые сигнализируют о: срабатывании пневматического тормоза (оранжевая), срабатывании реле перегрузки (зелёная), открытых дверях (белая). Длина вагонов метро может варьироваться: например на советских и российских вагонах серии А, Б, В, Г, Д, Е и его модификаций, 81-717/714 и 81-720/721 «Яуза» она составляет 19-20 метров; а сочленённых двухсекционных вагонов модели 81-740/741 «Русич» — 27—28 метров. Электропоезда метрополитена получают электричество от сети постоянного тока — как правило, от третьего (контактного) рельса, напряжение которого составляет 750—900 Вольт. Постоянный ток получают на подстанциях из переменного тока с помощью выпрямителей. Ширина колеи метрополитена различна в разных странах и, как правило, соответствует принятой ширине колеи железнодорожного транспорта, в России и странах СНГ — 1520 мм. В метро также эксплуатируются контактно-аккумуляторные электровозы, автомотрисы и мотовозы для возможности перемещения вагонов и путевых машин и рабочих в ночное время, когда напряжение на контактном рельсе отключено.
Управление подвижным составом может быть и полностью автоматизировано: впервые в мире такие поезда были применены в лондонском метрополитене на линии «Виктория» в начале 1970-х. Автоматизация позволила повысить скорость движения поездов и на 25—30 % сократить обслуживающий персонал.

Инфраструктура
Станции
Станции используются для посадки и высадки пассажиров из вагонов. Подземные, а также надземно-эстакадные станции сообщаются с поверхностью с помощью вестибюлей, турникетов, эскалаторов (или просто лестничных сходов, а кое-где также лифтов для инвалидов), осуществляющих пропуск пассажиров.
Конструктивно станции бывают колонного, пилонного, односводчатого и смежных типов, а по расположению платформ относительно путей делятся на островные и береговые. Существуют многопутные и многоуровневые пересадочные станции.
Некоторые станции сооружаются закрытого типа со стенами и дверями — преимущественно стеклянными — между платформой и поездом.
Многие станции Московского, Петербургского, Пхеньянского, Стокгольмского и ряда других метрополитенов оформлены как дворцовые залы или просто как архитектурные и художественные новаторства.

Безопасность
Станции метрополитенов оборудуются лифтами для маломобильных пассажиров, тактильным покрытием для безопасности и помощи ориентации в пространстве пешеходам с нарушением зрения и платформенными раздвижными дверьми.

Тоннели
Довольно часто линии метро проложены в подземных тоннелях. Тоннели линий метро бывают двух- и однопутные. Двухпутные тоннели применяются в однотонельных схемах подземных линий метро.
Однопутные тоннели применяются в двухтонельных схемах подземных линий метро, в которых каждый путь линии метро пролегает в своём тоннеле. Двухтонельная схема на подземных линиях метро и, следовательно, однопутные метротоннели в настоящее время явно доминируют.
Чтобы избежать пересечений в одном уровне, тоннели пересекающихся подземных линий метро прокладывают на различной глубине.
В гористой местности тоннели (как двух-, так и однопутные) также могут применяться для участков линий метро, проходящих сквозь горы.

Эстакады
Зачастую наземные линии метро подняты на эстакады. На эстакадных линиях метро преобладают двухпутные эстакады. Парные однопутные эстакады используются сравнительно редко. Чтобы избежать пересечений в одном уровне:

эстакады пересекающихся линий в местах пересечений поднимают на разные высотные уровни;
если это возможно, одна из пересекающихся линий в месте пересечения опускается на уровень земли.
Эстакады также могут использоваться в качестве путепроводов там, где нужно обеспечить пересечение в разных уровнях наземной линии метро с:

автодорогой,
наземной трамвайной или наземной железнодорожной линией,
другой наземной линией метро.

Метромост
Метромост — мост, по которому проходит линия метрополитена. Этот мост отличается от обычного повышенной прочностью, так как поезда метро создают очень сильную вибрацию. В некоторых случаях применяют совмещённый метромост. Часто такой мост двухъярусный, — на верхнем ярусе располагается автомобильная или железная дорога, а на нижнем — линия метро (яркий пример — Нижегородский метромост). Но встречаются и одноярусные совмещённые метромосты, на которых пути лини метро проложены или вдоль краёв проезжей части автодороги или, наоборот, в середине моста, а проезжие части автодороги, соответственно, слева и справа от линии метро (например, Нагатинский метромост и Южный мост). Также есть станции метрополитена, расположенные на метромостах, например, Воробьёвы горы в Москве, Днепр в Киеве или Аметьево в Казани. На сентябрь 2013 года самый длинный метромост в мире (расчёт дан с эстакадами) эксплуатируется в Новосибирске между станциями «Речной вокзал» и «Студенческая» при это проходящий через станцию «Спортивная», расположенную на метромосту..

Путь и оборотный тупик
Движение поездов на станциях осуществляется по 1 и 2 станционным путям, называемым также главными станционными путями. 3 и 4 станционные пути за станцией, как правило, являются оборотными. 1 главный станционный путь соединён с 3 и 4 станционным, а следовательно 3 и 4 соединены с 2 главным станционным путём. Отсюда оборот с 1 на 2 главный путь осуществляется через 3 или 4 станционные пути. Они служат для оборота подвижного состава на конечных станциях, на центральных станциях линии в случае организации зонного движения или непланового прекращения движения по концевым участкам линии.
При наличии в тупиках пункта технического обслуживания (ПТО), в часы работы ПТО по графику подвижной состав оборачивается только по одному из двух оборотных путей 3 или 4. Другой оборотный путь используется для технического обслуживания (осмотр и мелкий ремонт) подвижного состава. Во время стоянки подвижного состава работниками ПТО снимается напряжение с контактного рельса при помощи разъединителя с ручным приводом. В ночное время на оборотных путях по графику осуществляется отстой подвижного состава. Машинисты выходят из тоннеля, поднимаются на поверхность и идут отдыхать в комнаты ночного отдыха локомотивных бригад, расположенные в непосредственной близости у станции. Один из оборотных путей бывает востребован и ночью, на нём оборачиваются мотовозы. В случае неисправности подвижного состава на линии в часы наиболее интенсивного движения — в «час пик» — по команде поездного диспетчера состав убирается с главных путей на один из оборотных путей ближайшей станции. В ночное время состав перегоняется в электродепо приписки.
Рассмотренная схема предполагает наличие двух оборотных станционных путей. Существует также схема организации движения на конечных станциях с одним оборотным путём, однако она используется реже и преимущественно на центральных станциях, где частота использования оборотного пути крайне низка.
Оборотные пути могут не строиться в случае низкой интенсивности движения. В этом случае оборот поездов осуществляется непосредственно по одному или по обоим главным станционным путям с использованием перекрёстного или косого съезда перед станцией. В этом случае состав либо сразу прибывает на неправильный путь, либо наоборот — отправляется с неправильного пути. Подобная схема используется в Новосибирском метрополитене на станциях Золотая Нива и Площадь Гарина-Михайловского и в самарском метрополитене на станциях Юнгородок, Российская и Алабинская. В Москве так организован оборот поездов на станциях Александровский Сад, Кунцевская, Международная Филёвской линии и Алма-Атинская.

Инженерный корпус
В инженерном корпусе метрополитена расположен центр управления движением поездов и работой всех технологических установок (электротехнических, связи и автоматики, сантехнических и др.), которые обеспечивают эксплуатацию метрополитена. Инженерный корпус оснащён всевозможными оборудованием и устройствами. В нём также находится управление работой метрополитена и аппарат разных служб. Компьютеры, находящиеся в инженерном корпусе, следят за слаженностью системы: интервалом движения поездов и т. д.

Электродепо
Электродепо в метрополитене — предприятие, эксплуатирующее и ремонтирующее подвижной состав метрополитена. Также оно используется для хранения поездов метро и специализированных поездов для осмотра путей.

Гейт
Гейт (англ. gate — ворота) — место соединения метрополитеновской и железнодорожной сетей. Гейты используются в основном для того, чтобы привезённые по железной дороге вагоны метро, железнодорожные рельсы и прочие грузы для метро доставить в метрополитен; при этом ходовые рельсы соединительной ветки плавно переходят в пути метрополитена, так как ширина колеи у них одинаковая. Чаще всего соединительные ветви с железной дорогой располагаются у электродепо метрополитена.

Двойное назначение
При проектировании большинства подземных метрополитенов учитывается необходимость обеспечения возможности использования их в качестве бомбоубежища для населения. Для этого, как правило, предусматривают оборудование станций и перегонов аварийными автономными системами фильтровентиляции, энерго- и водоснабжения, запасными выходами, системами герметизации станций и вентиляционных шахт (в том числе — автоматическими, от действия ударной волны взрыва, проникающей радиации, появления в воздухе отравляющих веществ и т. п.).
По действующим в России нормативам, метро должно обеспечивать укрытие населению в течение двух суток: предполагается, что за это время уровень заражения спадёт до значений, при которых будет возможна эвакуация населения за пределы пострадавшей территории. Вместе с тем, на практике исполнение этих требований зависит от пожеланий заказчика, в связи с чем новые станции Московского метрополитена оборудованы металлоконструкциями почти все, тогда как в постсоветском Казанском метрополитене системы обеспечения гражданской обороны из соображений экономии были установлены только на некоторых станциях (и вообще невозможны на одной наземной станции ввиду расположения на метромосту и на двух подземных станциях ввиду наличия стеклянных атриумов от поверхности). С другой стороны, современные технологии строительства подземных сооружений часто способны обеспечить адекватную защиту при сравнительно небольшой глубине заложения. Кроме того, во многих городах мира при строительстве подземных линий и станций гражданского метрополитена, в особенности глубокого заложения, также строятся расположенные на одном уровне с ними подземные объекты гражданской обороны и военного назначения, имеющие сбойки с тоннелями и станциями метрополитена.
В Москве существует отдельная правительственная линия Д-6, или более известная под неофициальным названием «Метро-2».

Аварии и теракты
Метрополитены, а также метрострой являются объектами повышенной опасности, уязвимыми для техногенных аварий и катастроф, а также терактов.
Известны случаи значительного затопления больших участков метро как природного характера (например, в Ленинградском и Петербургском в 1974 и 1995 годах и Пражском метро в 2002 и 2013 годах), так и рукотворного (например, в Берлинском метро в конце Второй Мировой войны).
Крупнейшими авариями стали пожар в Бакинском метро в 1995 году с 289 погибшими и пожар в метро Тэгу в 2003 году с 198 погибшими.
Тяжелейшими терактами с около 40 погибшими были взрывы в Лондонском метро в 2005 году и в Московском метро в 2004 и в 2010 годах, самым масштабным терактом с тысячами пострадавших — зариновая атака в Токийском метро в 1995 году. Для предотвращения терактов на станциях многих метрополитенов (в Китае — во всех, в России — пока только во многих) на входах в вестибюли установлены рамки металлодетекторов и детекторы взрывчатых средств, хотя известны случаи терактов и перед такими объектами защиты.

Метро в современной культуре
В печати
«Метро 2033», «Метро 2034», «Метро 2035» — серия постапокалиптических романов Дмитрия Глуховского, описывающая жизнь людей в московском метро после ядерной войны.
«Вселенная Метро 2033» — романы и рассказы других авторов, продолжающие и дополняющие романы Глуховского. Действие (кроме «Метро 2033: Север», «Метро 2033: Осада рая» и нескольких рассказов сборника рассказов «Метро 2033: Последнее убежище») происходит в метрополитенах разных городов Беларуси, Великобритании, Италии, России и Украины.
Повесть «Не прислоняться» Олега Дивова совместно с Максимом Рублёвым в «Живом Журнале».
«Метро» — бесплатная ежедневная газета, распространяемая в метрополитенах 19 стран мира. В России выходит в Москве — «Metro» и Санкт-Петербурге — «Метро в Санкт-Петербурге».

В музыке
Metro — венгерская рок-группа 1960-х.
«Метро» — московская рок-группа во главе с Юрием Царёвым, исполнявшая музыку в стилях новая волна и электроник-рок.
«Метро» — свердловская рок-группа во главе с Аркадием Богдановичем.
Группа M.E.T.R.O. — проект Касаева Артёма, вокалиста рок-группы «Оригами».
Альбом «Стать машинистом метро» группы Mashinistmetro (заглавная песня «Стать машинистом метро»).
Песня «Метро» группы «Високосный год».
Песня «Метро» группы «Данила Master».
Песня «Метро 2033» группы «Леон».
Песня «Мне в моем метро…» барда Булата Окуджавы.
Песня «В городе, где нет метро» группы «25/17».
Песня «42 минуты под землёй» Валерия Сюткина.
Песня «Линии Метро» группы «Время и Стекло».
Песня «Звёзды не ездят в метро» группы «Машина времени».
Песня «Metropolitan Mail» Гуфа.
Песня «Имени Ленина» Slim при участии Скин, Mesr.
Песня «Метро» Александра Пушного.
Песня «Метро» группы «Где моё лето».
Песня «Метро» группы «Друга Ріка».
Песня «Кольцевая» группы «Звери».
Песня «Станция Таганская» группы «Любэ»
Песня «Метро» группы «Чебоза».
Песня «Метрополитен» группы «Магнитная Аномалия».
Песня «Метро» группы Renaissance v.2.0.
Песня «Кантемировская» Noize MC.
Песня «Metro» группы System of a Down.
Песня «M» группы the Chemodan clan.
Песня «Метро» певицы Дакоты.
Песня «Метро» группы Tracktor Bowling.
Песня «На Безымянке» группы «Без Сна».
Песня «Поезд в сторону „Арбатской“» группы «Тараканы!».
Песня «Метро» группы «СЕtИ».
Песня «В метро» певицы Земфиры.
«Метро» — польский мюзикл.
«Метро» — российский мюзикл о шоу-бизнесе (1999).
Metro FM — российская радиостанция.

В кино
«Метро» — корейский фильм 2003 года.
«Метро» — российский фильм-катастрофа 2013 года режиссёра Антона Мегердичева.
«Подземка» (англ. Subway) — французский кинофильм 1985 года. Режиссёр Люк Бессон.
«Мёбиус (1996)» — аргентинский фантастический фильм про исчезновение состава с пассажирами в метро Буэнос-Айреса.

В компьютерных играх
Metro 2033 — постапокалиптический шутер, основанный на романе Дмитрия Глуховского «Метро 2033».
Metro: Last Light — игра по мотивам книги «Метро 2034», продолжение игры Metro 2033.
Metro Exodus — игра вдохновлена на романах «Метро 2033» и «Метро 2035», продолжает историю Metro: Last Light.
Mini Metro — игра-головоломка, выпущенная в 2015 году.

См. также
Словарь метротерминов

Комментарии
Комментарии
Примечания
Литература
Day, John R.; Reed, John. The Story of London's Underground (англ.). — 10th. — Harrow: Capital Transport, 2008. — ISBN 978-1-85414-316-7.

Ссылки

Мир метро Сайт о метрополитенах разных городов.
Всё о метрополитенах городов бывшего СССР.
Метроэнциклопедия Роберта Швандля (англ.).
Станции метро на территории бывшего СССР: архитектура, схемы линий, история, будущее.
Метроблог — новости метро и городского транспорта.
Метровагоны — вагоны метрополитена, авторы Олег Бодня, Михаил Березин и Михаил Кончиц.
Метро Битс (Metro Bits) — Various aspects of the world’s metros (англ.).
Метрополитены Архивная копия от 11 февраля 2007 на Wayback Machine на сайте «Железнодорожное кольцо».
Коллекция схем метро (по городам мира).
ru_metro — сообщество  «Метрополитен» в «Живом Журнале», которое ведут пассажиры метрополитена. Интересные наблюдения, истории, фотографии.
foto_metro — сообщество  «Фотографии метрополитена» в «Живом Журнале»
Сайт истории метрополитенов в схемах.

## Title: `Тоннель`

...

## Title: `Московский метрополитен`

Моско́вский метрополите́н им. В. И. Ле́нина (также моско́вское метро́ или мосметро́, в 1930-х годах — моско́вский метро́; с 13 мая (открыт — 15) 1935 года по 29 ноября 1955 года — им. Л. М. Кагано́вича) — рельсовый внеуличный (преимущественно подземный) скоростной городской общественный электротранспорт агломерации (в Москве и области — городах Красногорске и Котельниках и, в будущем, Мытищах); старейший и крупнейший на постсоветском пространстве и крупнейший в Восточной Европе. Московское метро в 2016 году было шестым в мире по интенсивности использования после Пекинского, Токийского, Шанхайского, Сеульского и Гуанчжоу; на конец 2023 года занимает 10-е в мире и первое в России и Европе место по длине и количеству эксплуатируемых линий. Первый в современной России, в котором к управлению электропоездом допущены женщины. Третий в мире (после Мадридского и Пекинского), имеющий две кольцевые линии. Московский метрополитен эксплуатирует ГУП «Московский метрополитен», с 2011 года подчиняющееся Департаменту транспорта и развития дорожно-транспортной инфраструктуры и Департаменту строительства Москвы.
Первая линия — Сокольническая — открылась 15 мая 1935 года, в момент запуска насчитывала 13 станций, имела длину 11,2 км и шла от станции «Сокольники» до станции «Парк культуры» с вилочным ответвлением от «Охотного Ряда» к «Смоленской». Метро состоит из 15 линий длиной около 481,4 км в двухпутном исчислении с 275 станциями, из которых более 40 являются памятниками культурного наследия.
Интервалы движения поездов в часы пик на некоторых линиях, составляющие 90 секунд, являются самыми короткими в мире. В феврале 2023 года на Кольцевой линии интервалы движения также впервые в мировой практике были снижены до 80 секунд.
Средняя скорость движения в тоннелях между станциями Московского метрополитена в 2021 году составляла 41,61 км/ч.
В соответствии с транспортной программой Москвы до 2030 года, анонсированной мэром Сергеем Собяниным 19 февраля 2024 года, за период с 2024 по 2030 год планируется открыть 3 новые линии — Троицкую, Рублёво-Архангельскую и Бирюлёвскую, а также 39 новых станций; из них в 2024 году были открыты 8 станций, в том числе первый участок новой Троицкой линии.

История
Неосуществлённые проекты
Первые предложения по созданию метро в Москве появились ещё в 1875 году, когда возникла идея проложить линию от Курского вокзала через Лубянскую и Трубную площади до Марьиной Рощи.
В 1902 году инженеры П. И. Балинский и Е. К. Кнорре предложили проект сметной стоимостью 155 млн рублей, по которому метро должно было соединить Замоскворечье с Тверской заставой подземной линией. Однако городская дума отклонила его, вынеся резолюцию: «Господам Кнорре и Балинскому в их домогательствах отказать…». Гласные думы усомнились в проработанности проекта, играло роль и существовавшее тогда трамвайное лобби (трамвай приносил тогда значительную часть прибыли в казну). Л. И. Ковалёв в пропагандистской книге «Метро. Сборник, посвящённый пуску Московского метрополитена» в феврале 1935 года без ссылок на архивные документы или другие источники сообщает, что о проекте Балинского некий «архиерей Сергий» якобы написал московскому митрополиту следующее письмо:

В том же 1902 году инженерами путей сообщения А. И. Антоновичем, Н. И. Голиневичем и Н. П. Дмитриевым был разработан проект Московской городской железной дороги. В отличие от проекта Кнорре и Балинского, он предусматривал как подземные (в центре города), так и надземные линии по грунту или на эстакадах — на окраинах. Планировались четыре радиальных линии и одна кольцевая по трассе Камер-Коллежского вала с пересадками.
В 1913 году Московская городская управа разработала свой проект подземной железной дороги, состоящей из трёх подземных диаметров: Таганско-Тверского (от Тверской заставы до Калитников); Арбатско-Мясницкого (от Каланчёвской площади до Брянского (Киевского) вокзала) и Виндавско-Замоскворецкого (от Виндавского (Рижского) вокзала до нынешней платформы ЗИЛ).
Известен детально проработанный проект электротехника М. К. Поливанова от 1916 года. Тоннели трёх подземных диаметров соединялись с путями магистральных железных дорог, пригородные участки которых должны были быть электрифицированы.
В 1923 году к реализации проекта московского метро сметой в 30 млн рублей на правах концессии приступила немецкая компания Siemens Bauunion GmbH, — как и многие инфраструктурные проекты периода НЭПа московский метрополитен был не муниципальным предприятием, а ПИИ. К 1925 году немецкий проект, включавший 80 км тоннелей и 86 станций, был готов. Однако денег на реализацию не нашли, и проект Siemens Bauunion GmbH остался на бумаге. В 1925 году был разработан проект Мясницкого радиуса, но он не был осуществлён. Согласно первому пятилетнему плану, к 1932 году предполагалось построить первую линию метро: центр — Каланчёвская площадь.

Осуществлённый проект
Советское время
К принятию решения о строительстве Московского метрополитена 15 июня 1931 года на Пленуме ЦК ВКП(б) после доклада первого секретаря Московского горкома партии Лазаря Кагановича сподвиг дорожный затор, целиком парализовавший движение всего транспорта 6 января того же года — от трамваев до извозчиков. Несмотря на это, сам Каганович видел решение транспортной проблемы в электрификации пригородных направлений и использование их в самой Москве, однако должен был подчиняться И. В. Сталину. Тем не менее, по сути реализацию этих планов можно наблюдать в современности в виде МЦД, в средствах массовой информации и пиар-кампании преподносящихся как метрополитен (а иногда и напрямую метрополитеном), однако по совокупности характеристик и технического оснащения им не являющихся, будучи связанными с ним единой системой оплаты проезда.
Положение о Метрострое утвердил 13 сентября 1931 года Совнарком РСФСР, а 2 октября — Совнарком СССР. Первым главой Метростроя стал Павел Роттерт. Основой проекта стал изменённый проект Siemens-Bauunion, предполагавший строительство открытым способом. Где это было невозможно (под домами, железными дорогами), наметили строительство тоннелей от вертикальных шахт.
В ноябре 1931 года начали строить первый опытный участок на Русаковской улице. В ходе проектирования возник спор о типе будущих станций метро, какие у них должны будут быть платформы: островные или береговые. Решили остановиться на трёхсводчатой станции с островной платформой. Для подъёма пассажиров на поверхность планировали использовать эскалаторы. Московский инженер В. Л. Маковский обосновал возможность и необходимость прокладки в сложных условиях московских грунтов тоннелей глубокого заложения.
В 1933 году утвердили технический проект первой очереди метрополитена, тогда же трест Метрострой начал основные строительные работы. Маршрут первой очереди разработали, исследовав пассажиропоток московского трамвая: подземкой решили повторить самые напряжённые его маршруты.
Для строительства по-прежнему не хватало людей. По решению созванного 29 декабря 1933 года пленума Моссовета за каждым московским предприятием закреплялся отдельный участок линии метрополитена. Рабочие московских заводов и фабрик в добровольно-принудительном порядке направлялись на многотысячные субботники. Среди москвичей строительство метро, куда уходило огромное количество средств, вызывало раздражение.
При прокладке участков метрополитена применялись различные способы. Сооружение участков от станции «Сокольники» до станции «Комсомольская» и от станции «Библиотека имени Ленина» до станции «Парк культуры» велось открытым способом. Тоннели между станциями «Улица Коминтерна» (с 1990 года — «Александровский сад») и «Смоленская» возводили траншейным способом. На участке глубокого заложения от «Охотного Ряда» до «Площади Дзержинского» (с 1990 года — «Лубянка») была применена щитовая проходка. При прокладке переходных (от мелкого заложения к глубокому) участков от «Комсомольской» до «Красных Ворот» и от «Охотного Ряда» до «Библиотеки имени Ленина», где встретились плывуны, строители использовали кессонный способ с применением сжатого воздуха, замораживание и силикатизацию грунтов, искусственное водопонижение. 15 октября 1934 года по только что построенному участку линии был пущен первый поезд, состоявший из двух вагонов.

Вопрос участия заключённых в строительстве Московского метрополитена
В начале XXI века стало бытовать мнение, что к строительству первой очереди московского метро привлекали заключённых. Оно не подтверждается фундаментальной работой немецкого социолога Дитмара Нойтатца «Московское метро от первых проектов до великой стройки сталинизма», в которой он исследует социальный состав строителей метрополитена.

Особенности проектирования
Первоначально Народный коммиссариат путей сообщения РСФСР потребовал строительство линии по железнодорожному габариту для возможности проезда по путям метрополитена пригородных поездов (по аналогии с линией Метропо́литен в Лондоне), однако позже это было признано нецелесообразным и отменено.
Участки от «Сокольников» до «Комсомольской», от «Библиотеки имени Ленина» до «Парка культуры» и от «Александровского сада» до «Смоленской» сооружались открытым способом. На участке глубокого заложения от «Охотного Ряда» до «Площади Дзержинского» использовали английский метод щитовой проходки.
К проведению доводочных работ, — отделочно-декоративных, прокладки кабельной сети и др., — привлекались квалифицированные европейские и американские рабочие и техники.
4 февраля 1935 года прошёл первый пробный поезд, а 6 февраля 1935 года Московский метрополитен сдали в эксплуатацию. Машинистом электропоезда с руководителями партии и правительства был Николай Алексеевич Крейцберг, ветеран гражданской войны, ранее участвовавший в пуске первых троллейбусов в Москве (в 1936 году был арестован органами НКВД как враг народа). Для рядовых москвичей метрополитен открыли 15 мая 1935 года.
Первоначально должен был открыться не 15 мая 1935 года, а 7 ноября 1934 года.
При пробном пуске первого поезда в феврале 1935 года (планировался в январе) не были готовы устройства СЦБ, отвечающие за безопасность движения.
Первым пассажиром Московского метрополитена стал инвалид русско-японской войны, 75-летний пенсионер Ю. Х. Забровский, который потом пришёл и на открытие станции «Площадь Свердлова», а также «Курская» и «Площадь Революции» в 1938 году. Работники метрополитена его уже узнавали и пригласили на открытие новых станций Горьковского радиуса. В кинохронике, смонтированной из съёмок с разных станций, кадры с Ю. Х. Забровским сопровождались закадровым комментарием, что это — герой труда с завода «Красный пролетарий» Пётр Николаевич Латышев, который купил 15 мая 1935 года в кассе открывшейся станции «Сокольники» билет № 1 серии «А». А газета «Рабочая Москва» 15 мая 1935 года также опубликовала фотографию Ю. Х. Забровского, именуя его Петром Латышевым.
Пусковой комплекс включал 11,2 км трассы, 13 станций и 12 составов. Первая очередь шла от станции «Сокольники» до станции «Охотный Ряд», далее разделялась на две части: одна шла до «Парка культуры», другая — на «Смоленскую». Вторая из них, ставшая затем Филёвской линией, в 1937 году дошла до станции «Киевская», пересекая при этом Москву-реку по мосту. До начала Великой Отечественной войны были открыты ещё две линии. В марте 1938 года Арбатскую линию продлили до станции «Курская» (теперь участок относится к Арбатско-Покровской линии). В сентябре 1938 года открылась Горьковско-Замоскворецкая линия — от станции «Сокол» до станции «Площадь Свердлова».
Во время Великой Отечественной войны метро использовали как бомбоубежище. За время авианалётов в метро родилось 217 детей.
15 октября 1941 года Л. М. Каганович приказал закрыть Московский метрополитен, в течение 3 часов подготовить предложения по его уничтожению как стратегически важного объекта. Метро предполагалось уничтожить, а оставшиеся вагоны и оборудование вывезти. Утром 16 октября 1941 года в день паники в Москве метрополитен впервые не был открыт. Это был единственный раз в истории московского метро, когда оно не работало. Однако увидев, к чему это приводит, Государственный комитет обороны признал ошибочным такое решение — через несколько часов приказ об уничтожении метро был отменён и в 14:12 на Кировско-Фрунзенскую линию подали напряжение, в 18:05 поступил приказ о возобновлении движения, а в 18:45 пошёл первый поезд. То же самое было и на тогдашнем Горьковском радиусе (нынешняя Замоскворецкая линия), но ввиду большего количества требовавшихся работ, движение на нём было вновь запущено только 17 октября.
В связи с тем, что Москва становилась прифронтовым городом, осенью 1941 года началась эвакуация оборудования метрополитена. 179 вагонов метро были отправлены в Андижан, и с октября 1941 года по май 1942 года парк подвижного состава составлял всего 105 вагонов.

Строительство третьей очереди Московского метрополитена началось в 1940 году, ещё до начала Великой Отечественной войны. В начале войны оно было заморожено, однако возобновилось в мае 1942 года, после отвода угрозы захвата Москвы. Были введены в строй два отрезка пути: в январе 1943 года — «Площадь Свердлова» — «Завод имени Сталина» (с 1956 года «Автозаводская») (с пересечением Москвы-реки в глубоком тоннеле, причём станции «Павелецкая» и «Новокузнецкая» были открыты позже, в ноябре 1943 года, а в январе 1944 года — «Курская» — «Измайловский парк» (с 2005 года «Партизанская») (4 станции). На 7 станциях, построенных в военное время, имеются памятные таблички «Сооружено в дни Отечественной войны».
После войны начали строительство четвёртой очереди метрополитена — Кольцевой линии и глубокой части Арбатской линии от «Площади Революции» до «Киевской». Кольцевую линию первоначально предполагали строить под Садовым кольцом. Первая очередь линии — от «Парка культуры» до «Курской» (1950) расположена как раз под Садовым кольцом. Позже решили строить северную часть Кольцевой линии за пределами Садового кольца, обеспечивая доступ к семи из девяти вокзалов столицы. Вторая очередь Кольцевой линии открылась в 1952 году («Курская» — «Белорусская»), а в 1954 году строительство линии было завершено.
Строительство глубокой части Арбатской линии было связано с началом холодной войны. Станции глубокого заложения должны были служить бомбоубежищами в случае ядерной войны. После завершения строительства линии в 1953 году часть линии (от «Калининской» до «Киевской») была закрыта, но в 1958 году открылась вновь как часть Филёвской линии.
С 1955 года в связи с постановлением ЦК КПСС и Совета министров СССР «Об устранении излишеств в проектировании и строительстве» упор в развитии метрополитена был сделан на увеличение темпов строительства за счёт удешевления строительства станций. На каждую станцию стали выделять определённую сумму, и в неё необходимо было уложиться. От дорогих индивидуальных проектов каждой станции стали переходить к дешёвым типовым проектам. Последними станциями, построенными в классическом сталинском стиле, стали «Фрунзенская» и «Спортивная», открытые 1 мая 1957 года.
В конце 1950-х и в 1960-е годы развивалась концепция радиусов, соединённых лишь с Кольцевой линией: в 1958 году открылся Рижский радиус, в 1962 году — Калужский, в 1966 году — Ждановский, в 1972 году — Краснопресненский. В 1971 и 1975 годах соответственно радиусы были объединены в диаметральные линии — Калужско-Рижскую и Ждановско-Краснопресненскую. С конца 1970-х по начало 1990-х введены Калининская (1979—1986) и Серпуховско-Тимирязевская (1983—1994) линии, которые были построены по подобному проекту, сначала как радиус от кольца с постепенным продлением в обратном направлении, через центр, с образованием нового диаметра. В середине 1980-х годов появляется концепция скоростных хордовых линий метро, ведущих в спальные районы и аэропорты за пределами МКАД. Позднее вследствие снижения финансирования метрополитена эти планы были отложены на неопределённый срок.
В последние годы существования СССР началось строительство Люблинской линии.

Постсоветский период
В январе 1992 года постановлением правительства России  метрополитен был передан в собственность города Москвы, ранее будучи в ведении Министерства путей сообщения СССР.
В 1992—1994 годах был достроен северный участок Серпуховско-Тимирязевской линии, от «Отрадного» до «Алтуфьева» с промежуточной станцией «Бибирево». В 1995 году открыли Люблинскую линию, запроектированную ещё в середине 1980-х. В середине 1990-х появились новые проекты развития скоростного транспорта Москвы: лёгкое метро, монорельс, мини-метро и скоростная транспортная система. Реализовано было два: так называемое «мини-метро» (вилочное ответвление с обозначением «4А» Филёвской линии) и монорельс. В 2000—2001 годах был достроен южный участок Серпуховско-Тимирязевской линии от «Пражской» до «Аннина», а в конце 2002 года открыли конечную станцию «Бульвар Дмитрия Донского», впервые выводившую метро за МКАД, оставаясь при этом на территории Москвы (район Северное Бутово). В 2003 году пущена Бутовская линия, в то время полностью расположенная за МКАД. В 2004 году была построена линия монорельса. Планировался проект мини-метро изначально как частное ответвление от станции «Киевская» Филёвской линии в сторону делового центра «Москва-Сити». Оно должно было иметь кривые меньшего радиуса, более крутые подъёмы и более короткие платформы по сравнению с обычным метрополитеном. Однако затем от мини-метро отказались, и в итоге в 2005 году было построено простое ответвление с двумя станциями: «Выставочная» и «Международная».

В 2003—2009 годах Арбатско-Покровскую линию была продлена от станции «Киевская» до «Митино», включив в неё участок Филёвской линии. На этой линии построена первая станция на территории Московской области и одновременно первая станция, построенная на частные деньги (компанией Crocus Group Араза Агаларова), — «Мякинино». В 2007—2010 годах Люблинско-Дмитровскую линию продлили через центр города до станции «Марьина Роща».
15 мая 2010 года Московскому метрополитену исполнилось 75 лет. В честь юбилея на всех станциях установили памятные доски с датой открытия станции и именами архитекторов. На заводе «Метровагонмаш» в г. Мытищи Московской области был изготовлен ретропоезд «Сокольники», стилизованный под первые вагоны типа «А». Это касается как внешнего вида, так и заводского обозначения — 81-717.5А/714.5А — в соответствии с литерой первых поездов.
В декабре 2011 года открыли южный участок Люблинско-Дмитровской линии от станции «Марьино» до станции «Зябликово».
В августе 2012 года вышла за пределы МКАД ещё одна линия — Калининская. Была открыта станция «Новокосино», некоторые выходы которой частично расположены в подмосковном Реутове. В декабре того же года открыли станции метро «Алма-Атинская» (Замоскворецкая линия) и «Пятницкое шоссе» (Арбатско-Покровская линия), ставшие конечными станциями этих линий. В ноябре 2013 года Таганско-Краснопресненская линия была продлена в районы Москвы за МКАД, с открытием станций «Лермонтовский проспект» и «Жулебино».
В январе 2014 года пущен участок Солнцевской линии от «Парка Победы» до станции «Деловой центр». В феврале Бутовская линия была продлена до пересадки с Калужско-Рижской линией с открытием станций «Лесопарковая» и «Битцевский парк». В августе на действующем перегоне между станциями «Щукинская» и «Тушинская» открылась станция «Спартак», заложенная ещё в 1970-х годах, но остававшаяся законсервированной на протяжении почти сорока лет. В декабре была открыта станция «Тропарёво», ставшая первым этапом продления Сокольнической линии.

В сентябре 2015 года открыли станцию «Котельники», имеющую выходы сразу в трёх городах. В декабре открыли станцию «Технопарк» на наземном перегоне между станциями «Автозаводская» и «Коломенская». В январе 2016 года метро пришло в Новую Москву с открытием станции «Румянцево» на Сокольнической линии. В феврале была открыта следующая за ней «Саларьево», ставшая двухсотой станцией Московского метро.

В марте 2017 года пущен второй участок Солнцевской линии от станции «Парк Победы» до станции «Раменки» с двумя промежуточными станциями. В декабре Замоскворецкая линия была продлена в северном направлении от станции «Речной вокзал» до станции «Ховрино», которая открылась для пассажиров в последний день уходящего года — 31 декабря. На данном участке также находится промежуточная станция «Беломорская», строительство которой временно приостанавливалось, в дальнейшем её открытие состоялось на действующем перегоне.
В феврале 2018 года введена первая (северо-западная) часть Большой кольцевой линии — будущей второй подземной кольцевой Московского метрополитена, от станции «Деловой центр» до станции «Петровский парк» с тремя промежуточными станциями. В марте была введена в строй вторая очередь северного радиуса Люблинско-Дмитровской линии с тремя станциями от «Петровско-Разумовской» до «Селигерской». В конце августа введён в эксплуатацию участок Солнцевской линии от станции «Раменки» до станции «Рассказовка» с семью станциями. В декабре состоялось открытие станции «Беломорская» на действующем перегоне между станциями «Речной вокзал» и «Ховрино» и продление северо-западного участка Большой кольцевой линии на одну станцию от «Петровского парка» до «Савёловской».
В начале июня 2019 года пустили первый участок новой Некрасовской линии от станции «Косино» до станции «Некрасовка», в составе которого 4 станции, протяжённость — 6,9 км. В конце месяца был введён в эксплуатацию участок Сокольнической линии от станции «Саларьево» до станции «Новомосковская» с тремя промежуточными станциями. В конце октября 2019 года Каховская линия полностью прекратила своё существование в связи с запланированной реконструкцией станций и последующей интеграцией в состав Большой кольцевой линии в 2022 году.
В конце марта 2020 года введён второй участок Некрасовской линии длиной 14,4 км от станции «Косино» до станции «Лефортово» с 6 станциями. 31 декабря того же года этот участок был продлён на одну станцию — до «Электрозаводской».
1 апреля 2021 года была запущена первая очередь западного участка Большой кольцевой линии, от станции «Хорошёвская» до станции «Мнёвники» с промежуточной станцией «Народное ополчение». Организовано вилочное движение от «Хорошёвской» в сторону «Мнёвников» и «Делового центра». 7 декабря открылись ещё сразу 10 станций — вторая очередь западного участка, а также юго-западный и частично открытый южный участок Большой кольцевой линии от станции «Мнёвники» до станции «Каховская» с промежуточными станциями «Терехово», «Кунцевская», «Давыдково», «Аминьевская», «Мичуринский проспект», «Проспект Вернадского», «Новаторская», «Воронцовская» и «Зюзино».
1 марта 2023 года Большая кольцевая линия полностью замкнулась, открылись оставшиеся станции: «Варшавская», «Каширская» (открыты после реконструкции), «Кленовый бульвар», «Нагатинский Затон», «Печатники», «Текстильщики», «Марьина Роща», «Рижская» и «Сокольники». Участок «Нижегородская» — «Электрозаводская», который раньше был частью Некрасовской линии, начал свою работу в БКЛ 20 февраля, раньше, чем эти станции.
6 сентября 2023 года ввели станции на юге Солнцевской линии «Пыхтино» и «Аэропорт Внуково».
7 сентября 2023 года открылась станция «Физтех» на севере Люблинско-Дмитровской линии и стала самой северной станцией. Также открылись станции «Лианозово» и «Яхромская».
5 сентября 2024 года на южном участке Сокольнической линии открылась станция «Потапово», которая стала её новой конечной. Она является первой в московском метро отапливаемой крытой наземной станцией. Это достигается за счёт установленного на ней вентиляционного оборудования.
7 сентября 2024 года, в день празднования 877-летия Москвы, в строй был введён первый участок новой Троицкой линии протяжённостью 8,3 км, на котором открылись четыре станции: «Новаторская», «Университет дружбы народов», «Генерала Тюленева» и «Тютчевская». 28 декабря 2024 года для пассажиров начали работу ещё три станции: «Корниловская», «Коммунарка» и «Новомосковская» на втором участке протяжённостью 6,7 км. 13 сентября 2025 года был открыт третий участок Троицкой линии от «Новаторской» до «ЗИЛ» протяжённостью 9,7 км, с четырьмя станциями: «Вавиловская», «Академическая», «Крымская» и «ЗИЛ». С последних двух можно сделать пересадку на одноимённые станции Московского центрального кольца.

Название
Метрополитен первоначально носил имя Л. М. Кагановича, с 29 ноября 1955 года носит имя В. И. Ленина, в 1990-х и 2000-х годах оно использовалось не на всех станциях, хотя официально вопрос о снятии имени Ленина никогда не возникал. В 2014 году с инициативой убрать имя Ленина из названия метрополитена и его станций выступало московское отделение партии «Яблоко». В 2016 году руководство метрополитена заявило о намерении вернуть на все станции метро, где проходит реконструкция, таблички с названием станций со словами «Метрополитен им. Ленина».

Логотип
Вместе с открытием метрополитена 15 мая 1935 года появился и его логотип — заглавная М в связке с надписью «МЕТРО». Однозначных сведений о его авторе нет, поэтому в их числе указываются архитекторы станций метро — Самуил Миронович Кравец, Иван Георгиевич Таранов и Надежда Александровна Быкова.
К 2013 году в обращении находилось более десятка различных вариантов логотипа, поскольку корпоративной идентичности у бренда московского метро не было. Поэтому в октябре 2013 года был объявлен официальный конкурс на разработку корпоративного имиджа метрополитена, который, однако, был закрыт через несколько часов после объявления. Похожий конкурс, организованный независимой платформой дизайн-конкурсов «DesignContest», прошёл более успешно, однако официальной реакции со стороны метрополитена не получил.
В сентябре 2014 года студией Артемия Лебедева был анонсирован стандартизированный логотип московского метрополитена. Основой для единой эмблемы послужила усреднённая форма логотипов, которые появлялись в разное время в стенах метрополитена. Дизайнером обновлённой буквы выступил Константин Коновалов, за год до этого поставивший на публичное обсуждение вопрос стандартизации символа метро. Логотип стал частью нового бренда всего московского транспорта.

Схема
Есть много разнообразных схем Московского метрополитена, как официальных, используемых в вагонах, на станциях и в вестибюлях; так и неофициальных, выполненных сторонними изготовителями. Оформление официальной схемы за время существования метрополитена в Москве неоднократно менялось, следуя за веяниями моды и дизайна различных периодов истории страны.
Первая официальная схема, опубликованная в 1935 году, содержала детальную информацию о расстоянии и времени проезда между станциями; на более поздних официальных схемах такие подробности уже не указывались. На этой схеме первая линия Московского метрополитена, имевшая вилочное движение, была изображена схематично, цветовое обозначение маршрутов не использовалось: все тоннели на ней изображены чёрным цветом, а станции — красным.
До 1958 года вагонные схемы были чёрно-белыми, а до 1970-х они были нанесены на условную карту Москвы, давая более точное понимание о географическом расположении станций, чем современные. Хотя карта на схемах присутствовала не всегда, но станции располагались на схеме так, как если бы она была изображена. После появления цветовых обозначений линий их палитра не менялась. Исключением стала смена Рижским радиусом жёлтого цвета на оранжевый после соединения с Калужским радиусом, это было вызвано тем, что сначала эти радиусы планировалось не продлевать в центр города, а оставить заканчивающимися на пересечении с Кольцевой линией.
В 70-х годах XX века произведено кардинальное изменение дизайна схем: Кольцевая линия на них обрела форму правильной окружности, а радиальные и диаметральные линии стали изображаться прямыми и ломаными линиями. В 1980-х выпускались официальные схемы, на которых Кольцевая линия была изображена в виде эллипса, однако позднее было решено вернуться к отображению этой линии в форме идеального круга.
В таком виде дизайн официальной схемы дошёл и до наших дней; в разное время менялась толщина линий, обозначения станций (кружками либо засечками) и пересадочных узлов, но суть оставалась неизменной: круглая Кольцевая линия и пронзающие её ломаные прямые диаметральных линий. Было несколько попыток вернуть в официальную схему географичность, но они не увенчались успехом по причине того, что на таких схемах центр города, заполненный станциями и пересадками, получался перегруженным и плохо различимым.
Со временем на официальной схеме метрополитена появились обозначения и другого, в том числе рельсового скоростного транспорта Москвы, технически с метрополитеном не связанного: Московского монорельса (до 2025 года), Аэроэкспрессов, автобусов до аэропортов и Московского центрального кольца, что объясняется единой системой оплаты проезда; из них монорельс эксплуатируется одним с метрополитеном предприятием. Кольцевая линия и Московское центральное кольцо на официальной схеме изображаются в виде идеального круга, что далеко от их реальной формы (как, впрочем, и всех остальных линий метрополитена). С 2013 года созданием и обновлением официальной схемы по заказу департамента транспорта Москвы занимается студия Артемия Лебедева, ставшая победителем конкурса и онлайн-голосования, проводившегося среди жителей города.

Пользование метрополитеном
Правила
Правила пользования Московским метрополитеном утверждены Постановлением Правительства Москвы № 844-ПП от 16 сентября 2008 года. Их общий перечень приведён на официальном сайте Московского метрополитена. Нарушение правил влечёт административную ответственность.

Оплата проезда
Проезд оплачивают бесконтактными билетами, бесконтактными смарт-картами или бесконтактными банковскими картами (в том числе с помощью Samsung Pay), пропуск на станции контролируется автоматическими турникетами. Недавно[когда?] для оплаты проезда в московском метрополитене появились силиконовые, а позднее и кожаные браслеты, а также керамические кольца. Некоторые станции оборудованы реверсивными турникетами, работающими как на вход, так и на выход. На старых турникетах типа АКП-74 и АКП-74М при попытке неоплаченного прохода звучат начальные такты полонеза Огинского. На протяжении истории также использовались бумажные билеты, проверявшиеся контролёрами, турникеты с приёмом монет, жетонов и карт с магнитной полосой; последний из введённых способов оплаты — посредством банковских карт, снабжённых «транспортным» бесконтактным чипом. С 1 февраля 2013 года введён в обращение универсальный проездной билет, действующий как в метро, так и на наземном общественном транспорте, а со 2 апреля 2013 года также введены электронный кошелёк «Тройка» (за один проход с карты списывается меньшая сумма, чем по обычному билету на 1-2 поездки) и билеты на 90 минут. Стоимость одной поездки с июня 2025 года составляет 80 рублей, по карте «Тройка» — 67 рублей за обычную поездку и 100 рублей по тарифу «90 минут», оплата банковскими картами обходится пассажирам в 74 рублей, по биометрии — 63 рублей. Для пресечения неоплаченного прохода через турникеты за турникетными линиями обычно дежурят сотрудники полиции, службы безопасности метрополитена и контролёры ГКУ «Организатор перевозок», имеющие права выписывать штраф и изымать незаконные льготные билеты.
В начале февраля 2019 года в мэрии подтвердили информацию о том, что для столичной подземки разрабатывается новая система оплаты, позволяющая технически вводить «зональные билеты», но подчеркнули, что новые дифференцируемые тарифы для метро в зависимости от дальности поездок вводиться пока не будут. Пока зональная система оплаты по билетам московского транспорта была введена только на маршрутах пригородных поездов — Московских центральных диаметрах, на которые действует бесплатная пересадка из метрополитена и обратно в течение 90 минут.

График работы
Метрополитен открыт для пассажиров с 5:30 до 1:00, за исключением следующих вестибюлей:
 Комсомольская — по будням с 17:00 до 20:00 наклонный ход в сторону Ярославского и Ленинградского вокзалов работает только на выход. В случае ремонта эскалатора в этом наклонном ходе устанавливается дополнительное время ограничений с 7:00 до 10:00 (работает только на вход)
 Калужская — в связи с выявленными ошибками проектирования перехода на станцию «Воронцовская» северный вестибюль работает только на выход.
 Воронцовская — в связи с выявленными ошибками проектирования перехода на станцию «Калужская» по будням с 18:00 до 19:00 восточный вестибюль работает только на выход.
 Охотный Ряд, Театральная, Площадь Революции (кроме восточного вестибюля), Библиотека имени Ленина, Арбатская (кроме западного вестибюля), Александровский сад, Боровицкая — в дни репетиций и самого Парада Победы с 19:00 (на генеральной репетиции и 9 мая — с 5:30) до прохождения колонн военной техники станции работают только на вход и пересадку; во время прохождения военной техники вход на станции также закрыт. Выход с указанных станций осуществляется строго по спецпропускам ФСО или Министерства обороны РФ.
 Сухаревская,  Текстильщики,  Кузьминки — в дни Парада Победы и его репетиций вводятся ситуативные ограничения на вход и выход в связи с изменением в 2022 году маршрута выдвижения военной техники и принудительным закрытием подземных переходов, в которые ведут выходы с указанных станций метро.
 Парк Победы — 9 мая вестибюль южного зала работает только на выход, с 21:30 до 0:00 закрыт вход в вестибюль северного зала через выход № 6.
 Парк Победы — 9 мая вестибюль северного зала (выход № 6) закрыт на вход с 21:30 до 0:00.
До 2015 года некоторые станции, расположенные ближе к местам ночной расстановки составов или выдачи поездов из электродепо, открывались на вход с 5:20. На небольшом количестве станций с двумя и более выходами ранее только один был открыт всё время, остальные же работали по сокращённому графику. С 8 сентября 2019 года кассы на некоторых станциях начинают работу в 5:00.

Ровно в 1 час ночи останавливаются некоторые эскалаторы для входа и пересадки пассажиров, все станции работают только на выход. В 01:03 с конечных станций отправляется последний поезд (со станции «Пятницкое шоссе» — в 1:04, со станции «Александровский сад» в сторону «Москва-Сити» — в 1:08, со станции « Деловой центр» — 1:15). У припозднившихся пассажиров, уже находящихся в метрополитене, ещё есть шанс уехать из центра города на окраину — последние поезда на радиальных линиях проезжают центральные станции в районе 1:20—1:40.
По некоторым праздникам (Новый год, Рождество Христово, Пасха, а в отдельные годы и День Победы, День города и т. д.) с середины 2000-х годов принималось решение о продлении времени работы Московского метрополитена (как правило, до 2:00). С 31 декабря 2016 года был отменён ночной перерыв в движении в новогоднюю ночь и в ночь с субботы на воскресенье, в которые отмечается День города, а в дни вечерних матчей чемпионата мира по футболу 2018 (начинавшихся в 21:00 по московскому времени) работа метро и МЦК продлевалась до 3:00.
Средний интервал между отправлениями поездов составляет 2,5 минуты, минимальный в часы пик — 90 секунд, максимальный может достигать 10 минут и больше (ночью в выходные, либо при сбоях, нештатных ситуациях и несчастных случаях). На малозагруженных линиях держатся интервалы порядка 4—5 минут. На станциях «Деловой центр» и «Москва-Сити» в «часы пик» поезда отправляются раз в 5—6 минут.
График движения поездов выполняется на 99,944 %.

Беспроводная связь и Интернет
Сотовая сеть покрывает бо́льшую часть станций Московского метро, связью обеспечиваются многие переходы, эскалаторные наклоны и перегоны. Наличие покрытия и уровень сигнала зависят от конкретной станции (перегона) и сотового оператора. Все перегоны Кольцевой линии обеспечены непрерывным покрытием[уточнить].
С марта 2007 года на трёх станциях Московского метро («Охотный Ряд», «Театральная» и «Площадь Революции») компания «Комстар-ОТС» предоставляет платную услугу беспроводного доступа в Интернет (Wi-Fi). В 2012 году пробное оснащение метрополитена оборудованием Wi-Fi производилось компаниями «большой тройки»: на перегонах Кольцевой линии был проложен излучающий кабель МТС, станции и перегоны участка Серпуховско-Тимирязевской линии от «Менделеевской» до «Боровицкой» были оснащены «Мегафоном», а в двух поездах Сокольнической линии тестировался сервис Wi-Fi от «Билайна». Впоследствии все три компании отказались от участия в конкурсе на оснащение всего метрополитена как от экономически невыгодного проекта.
Обеспечением услуги Wi-Fi на всех линиях метрополитена согласилось заняться ЗАО «МаксимаТелеком», а соисполнителем выступила компания «Энвижн Груп». На Каховской линии Wi-Fi запустили в сентябре 2013 года, на Кольцевой — в декабре. В течение 2014 года сервисом были оснащены оставшиеся линии метрополитена: Калининская (февраль), Сокольническая (март), Люблинско-Дмитровская (июль), Замоскворецкая (август), Калужско-Рижская (октябрь), Таганско-Краснопресненская (октябрь), Серпуховско-Тимирязевская с Бутовской (ноябрь), Арбатско-Покровская и Филёвская (декабрь), при этом ввод в эксплуатацию сети на станциях, открытых позднее этого срока, происходит со значительным опозданием — от нескольких месяцев до года. Услуга бесплатна и предоставляется только в вагонах, создавать точки доступа в вестибюлях и на станциях не планируется. В настоящее время бесплатный (на условиях договора, запрещающего любые блокировщики рекламы) Wi-Fi предоставляется на всех перегонах, кроме некоторых участков, открытых в 2018—2019 годах, и во всех поездах, кроме ретро-поезда «Сокольники», что сделано с целью исторической аутентичности — он является репликой первого в СССР вагона типа «А», построенного в 1934 году.
В 2015 году АО «МаксимаТелеком» запустило новый сервис целевой рекламы, который получил название Aura Place. Сервис позволит показывать рекламу только определённым пользователям в зависимости от станции и линии метро, от времени, а также от предполагаемого места работы и проживания пользователя.

Пассажиропоток
Пассажиропоток Московского метрополитена является одним из самых высоких в мире. По количеству пассажиров, перевозимых в год, он уступает только Токийскому, Шанхайскому, Сеульскому, метрополитену Гуанчжоу, Пекинскому и Шэньчжэньскому. В 2022 году среднесуточный пассажиропоток составил 5,649 млн чел. Затраты на перевозку одного пассажира в 2022 году — 71,75 рублей, что выше стоимости проезда. Доля метрополитена в перевозке пассажиров в Москве составляет 48 %.
Динамика перевозки пассажиров в Московском метрополитене:

Количество пассажиров в метро зависит от времени, час пик наблюдается с 8:00 до 9:00 и с 18:00 до 19:00. Наименьший пассажиропоток — с 0:00 до 1:00. В выходные пассажиров в метро почти в два раза меньше, чем в будни. По данным 2008 года, наибольший среднесуточный пассажиропоток был отмечен в декабре (7637,1 тыс. чел.), а наименьший — в январе (6156,5 тыс. чел.). Максимальная суточная перевозка 2008 года была зафиксирована 26 декабря — 9352 тыс. чел. Новый рекорд был зарегистрирован 26 декабря 2014 года — 9715,6 тыс. чел. Абсолютный рекорд перевозки пассажиров за всю историю был достигнут в дни празднования 850-летия Москвы, когда метрополитен за 6 сентября 1997 года перевёз 14 миллионов пассажиров.

По данным на 2008 год, наибольший пассажиропоток приходится на следующие участки линий метро:

«Красногвардейская» — «Павелецкая» Замоскворецкой линии
«Выхино» — «Таганская» Таганско-Краснопресненской линии
«Планерная» — «Улица 1905 года» Таганско-Краснопресненской линии
«Алтуфьево» — «Менделеевская» Серпуховско-Тимирязевской линии
«Новоясеневская» — «Октябрьская» Калужско-Рижской линии
С окончания Великой Отечественной войны по 2009 год пассажиропоток в московском метро системно рос, однако в первом квартале 2009 года снизился на 7 % по сравнению с первым кварталом 2008 года. По мнению действовавшего начальника Московского метрополитена Дмитрия Гаева, причиной снижения стал мировой финансовый кризис. Ещё большее снижение пассажиропотока в метро наблюдалось при карантинных мерах во время активной фазы ковида в 2020 году.
Плотность пассажиропотока Московского метрополитена в час пик достигает 7,7 человека на 1 м² площади вагона по данным 2010 года, что почти в два раза выше нормы.

Руководство
Руководитель с 2017 года — Виктор Николаевич Козловский

Заместитель руководителя — Юлия Юрьевна Темникова (род. 1 марта 1984, Москва) — советник заместителя мэра Москвы по вопросам транспорта и промышленности, заместитель начальника Московского метрополитена.
В сферу её деятельности входит информационная политика, развитие бренда, маркетинг и организация публичных мероприятий Департамента транспорта и развития дорожно-транспортной инфраструктуры города Москвы. С мая 2024 года ведёт авторскую программу «Москва едет» на телеканале «Москва 24».
С 2017 года — заместитель руководителя по развитию клиентских сервисов и работе с пассажирами, позже — начальник по внешним коммуникациям и маркетингу. В 2022 году Темникова назначена советником заместителя мэра Москвы в правительстве Москвы по вопросам транспорта (Максима Ликсутова), руководителем пресс-службы и направления маркетинга Департамента транспорта.

Эксплуатирующие, проектные и строительные организации
Эксплуатацию метрополитена осуществляет ГУП «Московский метрополитен» (полное название — Государственное унитарное предприятие города Москвы «Московский ордена Ленина и ордена Трудового Красного Знамени метрополитен имени В. И. Ленина»). Его начальником с 23 мая 2017 года является Виктор Козловский.
ГУП «Московский метрополитен» — исключительно эксплуатирующая организация, строительством и финансированием новых линий не занимается. В пассажирском тарифе инвестиционной составляющей нет (присутствует только составляющая на реконструкцию и модернизацию), строительство метрополитена ведётся только за счёт бюджета Москвы (В 2011 году планировалось выделить из бюджета 2,7 млрд руб. Из них 1,5 млрд рублей на выкуп земельных участков под строительство новых линий метро, а дополнительные средства на модернизацию систем вентиляции в подземке и закупку автоматов по продаже проездных билетов). Проектированием и строительством занимаются специализированные организации: Мосинжпроект, Метрогипротранс, Мосметрострой, Трансинжстрой и другие. По различным подсчётам, стоимость строительства 1 километра Московского метрополитена — от 2,2 до 7 млрд руб.
В Московском метрополитене имеются более 30 эксплуатационных служб и различных формирований, каждое из которых выполняет определённые функции.
Служба тоннельных сооружений периодически осматривает все элементы тоннелей, занимается удалением пыли и промывкой поверхностей, а также выполняет косметический ремонт станций и вестибюлей. Большое внимание уделяется обнаружению и ликвидации течей грунтовых вод. Служба пути отвечает за эксплуатацию и ремонт путей метрополитена, смену рельсов и стрелочных переводов. Для контроля за состоянием пути используются дефектоскопные вагоны. Службы пути и тоннельных сооружений выполняют работы ночью в специально отведённое ночное «окно» (с 0:30 до 4:30). Также «окно» отводится и днём по субботам, когда на одной из линий закрывается центральный участок. В это время движение поездов осуществляется только на крайних участках, ближайшие к закрытому участку станции работают как конечные.

В советское время в московском метро работали женщины-машинисты — Мишина, Блинова и ряд других. Когда Шарлю де Голлю показывали Московский метрополитен, он ехал в кабине машиниста, в котором был женский экипаж, и он был удивлён тем, что женщины работали машинистами — это были военные годы, и женщины заменяли мужчин. В постсоветское время в московском метро до 2014 года работала последняя женщина-машинист — Наталья Владимировна Корниенко. Из-за запрета, установленного Правительством РФ 25 января 2000 года, до 2021 года в Московском метрополитене не работали женщины-машинисты. Этот запрет был отменён 1 января 2021 года, и с 3 января 2021 года в московском метро вновь начали работать женщины-машинисты. Снятие запрета с января 2021 года по август 2023 года касалось только Филёвской линии, поскольку на ней эксплуатируются одни из самых современных моделей поездов — «Москва» «первой модификации» (81-765.2/766.2/767.2). Пересмотр законов также связан с оснащением линии современным оборудованием. В августе 2023 года запрет на управление электропоездов женщинами был снят для Некрасовской линии. К ноябрю 2024 года количество женщин-машинистов в московском метро выросло по сравнению с прошлым годом почти в два раза — до ста сотрудниц, включая помощниц машинистов.
Машинистов и других сотрудников Московского метрополитена готовят в специализированном Корпоративном университете транспортного комплекса (бывш. Учебно-производственный центр).

Линии Московского метрополитена и интегрированных с ним городских транспортных систем
Московский метрополитен, МЦК и МЦД входят в единую систему скоростного рельсового транспорта Москвы и связаны единой системой оплаты проезда, движение поездов контролируется Единым диспетчерским центром, однако при этом МЦК и МЦД технически с метрополитеном не взаимосвязаны и по совокупности характеристик, таких как оснащение инфраструктуры и железнодорожной сигнализации, являются «городской электричкой». Применительно к МЦД в пиар-кампании массово употребляется особый термин — «наземное метро», а некоторыми городскими СМИ МЦД напрямую называются Московским метрополитеном, в то время как у определения «метрополитен» не существует «наземной» классификации, поскольку он является внеуличным транспортом, пролегающим как по поверхности, так и над и под ней. МЦК позиционируется Департаментом транспорта как вторая кольцевая линия метрополитена, однако является переоборудованным под пассажирское движение МКМЖД, которое было открыто раньше метрополитена.

Названия и порядковые номера
Всем линиям даны названия и порядковые номера: 1 — 16 линиям метро и D1 — D4 линиям МЦД. МЦК обозначается как линия 14. Калининская линия обозначается номером 8, а Солнцевская линия — 8А, так как эти линии в перспективе планируется объединить в Калининско-Солнцевскую линию.
Кроме полных названий линий используются и их аббревиатуры, например, АПЛ — Арбатско-Покровская линия, ГЗЛ — историческое (Горьковско-Замоскворецкая, в 1943—1990 гг.) название Замоскворецкой линии, на уровне освещения в СМИ сокращение аббревиатурой применяется к Большой кольцевой и, в меньшей степени, Таганско-Краснопресненской линиям — «БКЛ» и «ТКЛ» соответственно.
У линий также существовали впоследствии неиспользованные проектные названия (до их открытия): так, в частности, Сокольническая линия именовалась Кировско-Фрунзенским диаметром, Мясницким радиусом и Мясницко-Усачёвским диаметром, Калужско-Рижская — Дзержинско-Калужской и Калужско-Дзержинским диаметром.
Дополнительным устоявшимся обозначением линий служат их цвета, традиционно используемые на схемах (теми же цветами линии обозначены и в таблице ниже); несмотря на наличие официальных названий линий, немалому числу людей выражение «красная ветка метро» скажет едва ли не больше, чем «Сокольническая линия», а говорить «серая ветка» можно быстрее и легче, чем запомнить и проговаривать официальное название «Серпуховско-Тимирязевская».
Для каждой линии установлена нумерация главных путей в зависимости от направления, применяемая в технических документах метрополитена, а с 1999 года — и на указателях, при этом до 2003 года нумерация главных путей в залах кросс-платформенных пересадок не повторялась (на «Каширской» пути Каховской линии получили номера 3 и 4).

Линии метрополитена
Сеть Московского метрополитена состоит из 15 линий (без учёта Московского центрального кольца и Московских центральных диаметров). Большинство линий московского метро проходит через центр города, за исключением Бутовской, Солнцевской, Большой кольцевой, Некрасовской и Троицкой линий, расположенных на окраинах. Кольцевая линия соединяет все диаметральные линии.

Технические параметры и эксплуатационные особенности
Московское метро использует ту же ширину колеи, что и обычные железные дороги в России — 1520 мм, однако габарит приближения строений тоннелей и наземных сооружений у него значительно меньше. Для подачи тока используется нижний боковой третий (контактный) рельс с токосъёмом снизу, на который подаётся напряжение 825 В постоянного тока (на шинах подстанций — не более 975 В, на токосъёмнике вагона — не менее 550 В).
Бо́льшая часть путей и станций находится под землёй, однако есть и исключения. Так, Филёвская линия имеет длинный наземный участок от станции «Студенческая» до станции «Кунцевская» с семью наземными станциями, а четыре из семи станций Бутовской линии расположены над землёй на эстакадах. Наземные участки также есть на Таганско-Краснопресненской, Сокольнической, Замоскворецкой и на Арбатско-Покровской линиях. Некоторые наземные участки, например на юго-западе Сокольнической линии, снабжены крытым навесом. Станции «Мичуринский проспект» и «Пыхтино» Солнцевской линии имеют панорамные открытые путевые стены с видом на улицу и являются полуподземными.
Типичный путевой тоннель — однопутный, круглый в сечении с внутренним диаметром 5,1 м и внешним диаметром 6 м (в первых очередях — 5,5 м) или прямоугольный с внутренними размерами 4,16×4,4 м. Подземные двухпутные тоннели с прямоугольным сечением имеются на Сокольнической линии (участки «Преображенская площадь» — «Красные Ворота» и «Библиотека имени Ленина» — «Парк культуры») с разрывами на однопутные перед станциями, а также у порталов многих тоннелей других линий перед выходом на поверхность на наземные участки, а двухпутные тоннели с круглым сечением без разрыва двухпутного движения по станциями — на Большой кольцевой линии (участки «Аминьевская» — «Мнёвники» и «Текстильщики» — «Каширская»).
Участок Большой кольцевой линии длиной 3,3 км из 3 станций: «Каширской» (кросс-платформенная пересадочная), «Варшавской» и «Каховской» с 11 августа 1969 до 8 февраля 1985 года был частью Замоскворецкой линии, с 9 февраля 1985 до 19 ноября 1995 — её же вилочным ответвлением, а с 20 ноября 1995 до 25 октября 2019 года составлял отдельную линию — Каховскую. В составе Большой кольцевой линии он был повторно открыт 1 марта 2023 года (станция «Каховская» раньше остальных — 7 декабря 2021) после длительной реконструкции.
Также в московском метро есть шесть открытых метромостов. Из них четыре пересекают Москву-реку (Смоленский, Лужнецкий, Нагатинский и Митинский, из которых Смоленский — первый построенный в СССР), 1 — Яузу (Преображенский) и 1 — Ликову (Ликовский). Помимо них, имеется единственный четырёхпутный метромост через реку Лихоборку (в отличие от вышеперечисленных, находящийся на территории электродепо ТЧ-19 «Лихоборы», а не посреди линии), два средних пути которого накрыты металлическим коробом, а два крайних открыты. Кроме того, существуют шесть крытых метромостов, скрытых земляными насыпями, — например, Медведковский, представляющий собой тоннель, проходящий над Яузой.
Большинство линий Московского метрополитена рассчитаны на эксплуатацию поездов, состоящих из восьми стандартных 19-метровых вагонов. На линиях, как правило, эксплуатируются поезда из максимально возможного количества вагонов. Исключение составляют:

Сокольническая линия, рассчитанная на эксплуатацию восьмивагонных составов стандартной длины. На линии ходят составы как из семи (на такое количество рассчитано обслуживающее её электродепо ТЧ-1 «Северное»), так и из восьми стандартных вагонов. На линии также используются пятивагонные поезда модели «Русич» (двухсекционные вагоны, имеющие между собой сочленённое соединение, и, ввиду двухсекционной конструкции, один вагон «Русича» в полтора раза длиннее стандартного вагона), семивагонные именные составы ретропоездов «Сокольники» и «Красная стрела»;
Кольцевая линия, рассчитанна на эксплуатацию семивагонных составов. На линии используются составы из семи стандартных вагонов;
Филёвская линия, рассчитанна на эксплуатацию шестивагонных составов, за исключением станций «Александровский сад», «Арбатская» и «Смоленская», построенных в составе первой очереди, и станции «Киевская», которая была построена «отдельно» от остальной линии. Эти станции способны принять составы из восьми вагонов, поскольку находились на вилочном ответвлении Кировского диаметра (ныне — Сокольнической линии) от «Охотного Ряда», изначально рассчитанного на составы из восьми вагонов. На линии ходят составы из шести стандартных вагонов;
Арбатско-Покровская линия, рассчитанна на эксплуатацию семивагоных составов стандартной длины или пятивагонных «Русичей». На линии ходят «Русичи» из пяти вагонов (десяти соединённых между собой секций); также существовал период эксплуатации электропоездов «Ока» в семивагонной составности;
Бутовская линия, где ходят «Русичи» из трёх вагонов (шести секций), а также рассчитана на четыре стандартных вагона.

Организация вилочного движения
В Московском метрополитене организовано вилочное движение на Филёвской линии по маршрутам от «Александровского сада» до «Кунцевской» и от «Александровского сада» до «Москва-Сити» (с 10 сентября 2005 года). Эти маршруты с 2020 года обозначаются как 4 и 4А соответственно.
Ранее вилочное движение осуществлялось, проектировалось и планировалось по следующим линиям:

Сокольнической (с открытия до 13 марта 1938 года);
Замоскворецкой (с 9 февраля 1985 года до 19 ноября 1995 года, а для части рейсов до 30 марта 2019 года);
Арбатско-Покровской (в 2017 году часть поездов следовала от станции «Партизанская» до станции «Раменки» Солнцевской линии);
Большой кольцевой линии в виде станций «Деловой центр» и «Шелепиха» от станции «Хорошёвская» с 7 декабря 2021 года по 22 июня 2024 года, далее закрывшись для включения в Рублёво-Архангельскую линию;
Проектировалось на Калужско-Рижской линии по схеме «Свиблово» — «Бабушкинская» — «Свиблово» — «Лосиноостровская», под которое на перегоне «Свиблово» — «Бабушкинская» оставлен задел;
Планировалось на Таганско-Краснопресненской линии от станции «Полежаевская» в Серебряный Бор (в современности там планируется станция Рублёво-Архангельской линии). Об этом, в частности, свидетельствует нестандартное трёхпутное устройство «Полежаевской».

Перспективы развития
К 2032 году планируется продлить Сокольническую линию на 8,8 км за станцию «Бульвар Рокоссовского» до Ярославского шоссе. На новом участке, который пройдёт под национальным парком «Лосиный остров», планируется построить две станции — «МГСУ» и «Ярославскую». Геологические изыскания на трассе будущего участка проводились в октябре 2024 года. В июне 2025 года началась подготовка проекта планировки территории.
Новый перегон станет аналогом перегона «Крылатское» — «Строгино» на Арбатско-Покровской линии с эвакуационным выходом в виде технической платформы «Троице-Лыково» с предполагаемой длиной перегона 5 км, требующейся для протяжённых участков пути между станциями: перегон «Крылатское» — «Строгино» является самым длинным в московском метро — 6,6 км, время поездки по нему составляет 7 минут.
Также существуют планы по объединению Некрасовской линии с Троицкой, Рублёво-Архангельской с Бирюлёвской.

Лёгкое метро
В 2001 году началась проработка проекта лёгкого метро в составе системы Московского метрополитена. Было решено провести линии в «спальные районы», остро нуждающиеся в высокоскоростном транспортном сообщении.
Изначально предполагалось сооружение эстакад с кривыми предельно малого радиуса и эксплуатация сочленённых двухвагонных составов «Яуза», однако позже было решено начать создание нового типа вагонов специально для наземных линий метрополитена. Допустимые радиусы кривых на лёгком метро определены в 150 м согласно СНиП (строительные нормы и правила). Для сравнения: допустимые радиусы кривых для обычного метро определены в 200 м.

Метро-2
В Москве существуют засекреченные правительственные линии метро мобилизационного назначения, частично интегрированные с гражданским метрополитеном и именуемые «спецветками». В отличие от линий метрополитена и связывающих их служебно-соединительных веток, они не относятся к системе пассажирского транспорта Москвы и находятся в юрисдикции не самого метрополитена, а Главного управления специальных программ Президента Российской Федерации (ГУСП). В 1990-х годах с подачи писателя Владимира Гоника они стали широко известны как «Метро-2», при этом из-за их режимности и недостатка официальной информации открытые сведения о них стали предметом искажений и городских легенд, а в СМИ наряду с достоверной информацией были широко растиражированы недостоверные и сильно преувеличенные сведения о количестве и трассировке секретных линий. По информации из рассекреченных архивных правительственных документов 1960-х годов, достоверно известно о существовании и частично о трассировке двух спецветок — кремлёвском объекте № 100 с узкоколейкой и «Ветке» на юго-запад Москвы с путём метро стандартной колеи.
Более известной как Метро-2 является «Ветка» (проектное обозначение — ЧЗ-1090), которая также широко упоминается в ряде источников как «Д-6». Она служит для эвакуации правительства и военных из кремлёвского подземного комплекса 103 и объектов Министерства обороны на Знаменке и Фрунзенской набережной к высокозаглублённым подземным спецобъектам в Раменках, над которыми планировалось возведение нового комплекса правительственных зданий Дворца Советов, а также к дальнему пункту воздухозабора метрополитена у платформы Матвеевское, служившему резервной точкой эвакуации из метрополитена на окраины города в случае ядерного удара по центру. Большая часть системы была построена со второй половины 1950-х до начала 1970-х годов по заказу КГБ и Министерства обороны СССР; в 1980-х годах были построены дополнительные участки. Строительство отдельной подземной транспортной системы потребовалось потому, что участки мелкого заложения Сокольнической линии, как и Лужнецкий метромост, в случае ядерной войны были бы разрушены, делая невозможным сообщение и эвакуацию первых лиц государства через гражданское метро, которое к тому же было бы заполнено укрывающимися москвичами, что затрудняло движение поездов. До 1966 года система была частью Московского метрополитена, но позже была полностью передана на баланс КГБ (а после распада СССР — ГУСП), а первоначально построенный участок однопутного тоннеля от станции «Спортивная» стал соединительным между Сокольнической линией и «Веткой» и был оснащён закрывающимися воротами.
В тоннелях «Ветки» проложен путь стандартной колеи, но отсутствует контактный рельс, а рельсы утоплены в бетонные плиты для возможности движения автомобилей. Ввиду своего эвакуационно-служебного назначения система является однопутной. Станции представляют собой небольшие платформы в перегонном тоннеле с выходами к спецобъектам, по типу технических платформ в оборотных тупиках обычного метрополитена. Глубина заложения в среднем существенно превышает глубину гражданских линий метро, в частности в районе станции Университет и объектов в Раменках она составляет 189 м. Из-за отсутствия контактного рельса в системе эксплуатируются специальные дизель-поезда и автомотрисы (АС1А, ДПС, РА1, 81-730.05), ранее также эксплуатировались составы из контактно-аккумуляторных электровозов Л в сцепе со стандартными пассажирскими электровагонами Еж6. Эксплуатируемый подвижной состав системы находится на балансе войсковой части 95006 (Служба специальных объектов ГУСП), расположенная в Раменках недалеко от Матвеевского воздухозабора. По заказу этой войсковой части Московский метрополитен и завод «Метровагонмаш» осуществляют техническое обслуживание и ремонт подвижного состава «Ветки», для чего он временно передаётся в депо обычного метрополитена по соединительному тоннелю. Также по заказу этой воинской части Корпоративный университет Транспортного комплекса Московского метрополитена осуществляет подготовку машинистов поездов.

Кремлёвский объект № 100 был построен с 1951 по 1960-е годы и представляет собой транспортно-пешеходный тоннель, в котором был проложен узкоколейный рельсовый путь. Линия соединяет Московский Кремль (объекты № 1а и 25), бункер № 101 под стилобатом непостроенного административного здания под парком Зарядье, здание Администрации президента России (ранее — ЦК КПСС) на Старой площади и Здание органов госбезопасности на Лубянке (объект № 201) и на своих окончаниях переходит в рабочие помещения. Протяжённость тоннеля составляет около 2 км; диаметр — 6 метров, как и у метрополитена; глубина заложения — не менее 31 метра (под Кремлём — 55 м за счёт расположения под холмом). Система имеет сбойки с перегонными тоннелями Арбатско-Покровской, Замоскворецкой и Сокольнической линий, также в середине 1960-х годов строилось его пешеходное соединение с «Веткой» ЧЗ-1090, но более поздних точных сведений о данном объекте не обнаружено. Точные сведения о подвижном составе отсутствуют — известно, что первоначально в системе эксплуатировался 12-местный электровагон, позднее руководитель протокола президентов СССР Владимир Шевченко в своей книге «Повседневная жизнь Кремля при президентах» упоминал данную систему и эксплуатирующийся в ней «трамвайчик».

Другие линии внеуличного рельсового скоростного транспорта
Московское центральное кольцо
В 2012 году началась реконструкция Малого кольца Московской железной дороги под пассажирскую эксплуатацию. Изначально кольцо использовалось для грузовых перевозок между всеми десятью магистральными железнодорожными направлениями столицы, однако после реконструкции на первом и втором главных путях было организовано пассажирское движение. Линия пассажирских электропоездов представляет собой частично интегрированную с Московским метрополитеном (пересадки и частично система оплаты проезда) систему городской электрички — аналога немецкой модели S-Bahn. Помимо основного названия Московское центральное кольцо (МЦК) она получила неофициальное название «Вторая кольцевая линия» и порядковый номер .
Московское центральное кольцо открылось для пассажиров 10 сентября 2016 года. Линия представляет собой кольцо, состоящее из 31 станции, из них 6 имеют «прямые» (без необходимости выхода на улицу) переходы на станции метро. В течение месяца со дня открытия проезд по МЦК был бесплатным для всех пассажиров.

Московский монорельс (закрыт)
В систему Московского метрополитена входила монорельсовая дорога (эксплуатировалась одним с ней предприятием). Участок монорельсовой дороги длиной 4,7 км с шестью станциями соединял станции метрополитена «Тимирязевская», «Фонвизинская» и «ВДНХ». Первые поездки в «экскурсионном» режиме начались 20 ноября 2004 года, в полноценном режиме система заработала с 10 января 2008 года.
Технологически монорельс, как и метрополитен, был отделён от любых других видов городских транспортных инфраструктур, поездка на монорельсе изначально требовала отдельной оплаты. С 1 января 2013 года все виды проездных билетов для проезда на метрополитене также действовали и для оплаты проезда на Московской монорельсовой транспортной системе. При этом при осуществлении перехода со станций метрополитена на станции монорельса «ВДНХ» — «Выставочный центр», «Фонвизинская» — «Улица Милашенкова», «Тимирязевская» — «Тимирязевская» и обратно в течение 90 минут с момента входа на станцию дополнительная поездка с билета не списывалась.
На схемах в вагонах метрополитена система первоначально обозначалась логотипом М1, а с декабря 2015 года номером 13.
23 января 2017 года на линию был возвращён «экскурсионный» режим движения, поезд ходил раз в 30 минут. Поезда отправлялись с двух конечных станций с 8:00 до 20:00.
28 июня 2025 года монорельс завершил свою работу.

Станции Московского метрополитена
В Московском метрополитене 275 станций. Из них 265 расположены на территории Москвы (в том числе 11 — на территории Новой Москвы), 2 — в Московской области («Мякинино» в г. Красногорске), «Котельники» в г. Котельники), 1 — «Новокосино» — с выходами в г. Реутов. Многие станции или однократно, или по нескольку раз меняли свои названия («Александровский сад» и «Охотный Ряд»). Бо́льшая часть станций — подземные; 2 станции — полуподземные («Мичуринский проспект» и «Пыхтино»), то есть одной половиной они подземные, другой — на поверхности, 14 — наземные (в том числе 4 — крытые) и 5 — надземные (на эстакадах и мостах). Из подземных станций 83 глубокого заложения, 160 (включая «Мичуринский проспект») — мелкого. Глубокие станции по конструкции разделяются на трёхсводчатые пилонные (61), трёхсводчатые колонные (14), трёхсводчатые колонно-стеновые (6), трёхсводчатые колонно-пилонные (1, «Семёновская») и односводчатые (1, «Тимирязевская»). Станции мелкого заложения по конструкции разделяются на пятипролётные колонные (1, «Нижегородская»), четырёхпролётные колонные (1, «Александровский сад»), трёхпролётные колонные (89), трёхсводчатые колонные (1, «Волоколамская»), двухпролётные колонные (27), односводчатые (33) и однопролётные (4, «Волжская», «Марьино», «Улица Старокачаловская» и «Ольховая»).
Большинство станций Московского метрополитена имеют одну островную платформу и два пути, 18 станций имеют береговые платформы. Шесть станций имеют платформы, расположенные в кривой (в повороте). Две подземные станции («Партизанская» и «Полежаевская») имеют в одном зале по три пути и две островные платформы, а одна («Нижегородская») — четыре пути и две островные платформы. Одна наземная станция («Кунцевская») имеет три станционных пути, все из которых задействованы для пассажирского движения. Шесть подземных станций состоят из двух залов — почти все они кросс-платформенные, кроме «Улицы Старокачаловской», в каждом из залов которой находится по одному пути. При этом оформление залов кросс-платформенных станций различается.
Семь узлов станций имеют кросс-платформенной пересадку: пять из них — «Китай-город», «Третьяковская», «Каширская», «Петровско-Разумовская» и «Парк Победы» — подземные с двумя параллельными двухпутными одноплатформенными залами (при этом первые две из них расположены подряд на Калужско-Рижской линии), одна «Нижегородская» — подземная двухплатформенная четырёхпутная с единым залом, и ещё одна «Кунцевская» — наземная двухплатформенная трёхпутная. Два из этих узлов — «Третьяковская» и «Кунцевская» — трёхстанционные: кроме двух станций с кросс-платформенной пересадкой в каждый из этих узлов входят третья с обычными переходами, при этом на «Кунцевской» все три станции носят одинаковое название, а станция, пересадочная на обе «Третьяковские» — отличное («Новокузнецкая»). На «Кунцевской», в отличие от других перечисленных станций, кросс-платформенная пересадка возможна только в одном направлении: от центра со стороны Арбатско-Покровской линии и к центру со стороны Филёвской линии.
Две наземные станции, построенные на территориях депо, закрыты навсегда с продлением соответствующих линий мимо депо. Это «Первомайская», заменённая наземной же «Измайловской», и «Калужская», вместо которой была открыта одноимённая подземная станция.
12 станций были открыты на действующих перегонах, рекордсменом среди них по длительности существования в недостроенном состоянии является «Спартак», открывшаяся в 2014 году, на 39 лет позже участка «Октябрьское Поле» — «Планерная». Две станции из этих 12 были построены на действующих перегонах полностью — «Тверская» и «Технопарк» (все остальные станции, открывшиеся позже обеих соседствующих с ними, строились вместе с заделом на соответствующих участках). На перегоне «Крылатское» — «Строгино» располагается техническая платформа «Троице-Лыково», которую, теоретически, можно перестроить в полноценную станцию для пассажиров (так было задумано при проектировании участка «Парк Победы» — «Строгино», однако затем от этого отказались в пользу постройки станции «Липовая Роща» Рублёво-Архангельской линии).
Суммарно станции составляют 15 линий, обслуживаемых 45 пересадочными узлами. Из них один четырёхстанционный («Александровский сад» — «Арбатская» — «Библиотека имени Ленина» — «Боровицкая»), 9 трёхстанционных и 35 двухстанционных (в двух из них переход временно наземный).

Платформы снабжены громкой связью, по которой осуществляются объявления (о случающемся следовании поезда без остановки, отсутствии посадки на прибывающий поезд, о возможных технических проблемах на линиях метро и пр.). На многих станциях характерный звуковой сигнал извещает пассажиров о скором прибытии очередного поезда. На эскалаторах и на межстанционных переходах со сравнительно высокими пассажиропотоками звучат напоминания о правилах пользования метрополитеном, а также музыкальные темы и стихи знаменитых поэтов; в новогодние праздники звучат новогодние поздравления и новогодние песни; до 1 июня 2017 года там звучали сообщения рекламного характера.

Спуск на подземные станции метрополитена и подъём на надземные осуществляется при помощи эскалаторов и лестничных маршей. В большинстве случаев эскалаторы однокаскадные, трёх- или четырёхниточные. На станциях с 2003 года (кроме «Улицы Старокачаловской») устанавливаются лифты и подъёмники, причём на станциях мелкого заложения они связывают платформу через вестибюль с поверхностью, на станциях глубокого заложения только вестибюль и поверхность. На надземных станциях Бутовской линии для доступа к лифтам необходимо вызвать сотрудника метрополитена. До 2014 года подъёмник для людей с ограничениями опорно-двигательной системы существовал на станции «Алтуфьево».
Все подземные станции имеют наземные или подземные вестибюли, часто совмещённые с подуличными пешеходными переходами. Наземные вестибюли могут как представлять собой отдельные здания, так и быть встроенными в другие дома или объединёнными с ними (например, вокзалы). Подземные вестибюли часто имеют выход на поверхность в виде лестничных пролётов, вырезанных посреди улицы, которые иногда закрываются остеклёнными павильонами. Часть вестибюлей также играют роль перехода с одной станции на другую.
Две станции («Саларьево» и «Беломорская») имеют вестибюли, работающие только в одном направлении (только на вход или только на выход); на станции метро «Сокол» односторонней с 2006 года является только лестница восточного выхода, центральный вестибюль работает в оба направления, а северный совмещённый вестибюль станций метро «Комсомольская» с 2011 года по будням для Кольцевой линии с 16:30 (по пятницам с 15:30) до 20:00 работает только на выход (в случае ремонта эскалатора — также с 7:30 до 11:00 только на вход). Также односторонний выход на Рязанский проспект имеется на станции МЦК «Нижегородская» (до 27 марта 2020 года это был единственный вестибюль данной станции). Все остальные вестибюли работают в одностороннем режиме движения пассажиропотока только в случае временных изменений.
Вестибюли станций первой и второй очередей метрополитена многоуровневые, часто имеют наземный или встроенный павильон, далее расходящиеся лестничные марши, которые спускаются на промежуточные кассовый и турникетный уровни, и только потом доступ к платформам. Однако из-за неудобства такой схемы, начиная с третьей очереди, их сменили наземные вестибюли. В начале 1960-х, в связи с появлением типового проекта станций, была принята и новая типовая схема — подземный вестибюль, соединённый с подземными переходами. С тех пор наземные вестибюли стали исключением, и их строили только в отдельных случаях.

Площадь облицовки станций (всего) — 922,6 тыс. м², в том числе: мраморной плиткой — 392,4 тыс. м², гранитной плиткой — 100,3 тыс. м², разной плиткой — 230,5 тыс. м², прочей облицовкой — 199,5 тыс. м².
В облицовке около пятидесяти станций московского метро присутствуют различные окаменелости. Там можно найти раковины наутилусов, аммонитов и других доисторических моллюсков.

График роста количества станций
Количество станцийГод0501001502002503001935194819611974198720002013(1) Сокольническая(2) Замоскворецкая(3) Арбатско-Покровская(4) Филёвская(5) Кольцевая(6) Калужско-Рижская(7) Таганско-Краснопресненская(8) Калининско-Солнцевская(9) Серпуховско-Тимирязевская(10) Люблинско-Дмитровская(11A) Каховская(11) Большая кольцевая(12) Бутовская(15) Некрасовская(16) Троицкая.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729145:hover{pointer-events:none}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729146:hover{cursor:pointer;fill:rgba(0,0,0,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729147:hover{cursor:pointer;fill:rgba(82,130,235,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729148:hover{cursor:pointer;fill:rgba(255,199,56,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729149:hover{cursor:pointer;fill:rgba(255,132,111,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729150:hover{cursor:pointer;fill:rgba(140,225,196,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729151:hover{cursor:pointer;fill:rgba(41,174,82,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729152:hover{cursor:pointer;fill:rgba(193,212,255,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729153:hover{cursor:pointer;fill:rgba(159,143,213,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729154:hover{cursor:pointer;fill:rgba(238,198,225,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729155:hover{cursor:pointer;fill:rgba(193,144,47,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729156:hover{cursor:pointer;fill:rgba(178,185,194,1)}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729157:hover{cursor:pointer}.mw-chart-e4658f3bef4c02cf8c14c9b2b5d37c44b218e3fc963f06e191ba0e7dad6e62418b7a1b35c94f1f04a41f1416e181be92f6a09d6df3360119a318b7840ca384a6__zr142994-cls-729158:hover{cursor:pointer;fill:rgba(0,0,0,0)}

Объявления названий станций
Автоинформатор
Названия станций объявляются мужскими голосами при движении к центру Москвы, а на кольцевых линиях — по часовой стрелке; женскими — при движении в обратную сторону. Это сделано для ориентации незрячих и слабовидящих граждан. Такой вариант был предложен в 1984 году. Записи объявлений в информаторах сделаны Алексеем Россошанским (в вагонах объявляет с 2013 года на Таганско-Краснопресненской линии, с 2016 года на МЦК, с 2018 — началась замена Сергея Куликовских на всех остальных линиях, замена закончена в 2020) и Юлией Романовой-Кутьиной. До 2017 мужские объявления в вагонах Московского метро записывал Сергей Куликовских, последней его работой стало объявление Солнцевской линии в 2017—2018, в 2020 году он объявляет только на Калининской линии. В 2013 объявления для Таганско-Краснопресненской линии записала Екатерина Пясецкая, её голос не используется на других линиях, с 2016 постепенно заменялась на Юлию Романову-Кутьину и в 2020 заменена полностью, также она объявляла некоторые технические фразы на Серпуховско-Тимирязевской линии с 2013 до замены там информатора 2020. До 12 июня 2021 года в резервных головных вагонахТЧ-10  (20 маршрут) также действовал резервный автоинформатор старого образца (1990—2005 годов), хотя в основном перепрошивка автоинформатора на Калужско-Рижской линии была произведена в июне 2004 года, после передачи последнего состава в ТЧ-15 кассета была передана в частную коллекцию.
Во время Олимпийских игр 1980 года впервые в метрополитене объявления были продублированы на английский язык. В 2016 году на Таганско-Краснопресненской линии и МЦК после его открытия, а до конца 2017 — на всех линиях к проведению Кубка конфедераций и Чемпионата мира по футболу также звучали английские объявления в исполнении Светланы Екименко и Павла Новичкова. В 2021 году во время пандемии COVID-19 дублирование было отменено.

Акция «Голос метро»
В 2004 году стартовала акция «Голос метро». Вместо привычных голосов в метро зазвучали голоса знаменитых актёров. Объявления станций Сокольнической линии озвучили своими голосами Элина Быстрицкая, Татьяна Васильева, Екатерина Васильева, Галина Волчек, Валерий Гаркалин, Людмила Гурченко, Лев Дуров, Валерий Золотухин, Владимир Меньшов, Светлана Немоляева, Ольга Остроумова, Любовь Полищук, Константин Райкин, Нина Русланова, Валентина Талызина, Михаил Ульянов, Наталья Фатеева, Александр Ширвиндт, Борис Щербаков, Владимир Этуш, Сергей Юрский, Юрий Яковлев и Леонид Ярмольник. «Голоса метро» приветствовали пассажиров Сокольнической линии в День города, день рождения метрополитена и в новогодние праздники. После смерти Людмилы Гурченко акция «Голос метро» была приостановлена.
30 апреля 2015 года в связи с предстоящим празднованием 80-летия метро акция была возрождена под названием «Голоса метро». В обновлённом варианте акции станции на всех линиях метрополитена объявляются актёрами театра и кино, певцами и телеведущими.

В ходе акции не были переозвучены такие служебные объявления, как «Уважаемые пассажиры! Побыстрее выходите из вагонов! Побыстрее проходите в вагоны». Официально было объявлено, что акция продлится до конца мая 2015 года, но фактически информаторы почти во всех поездах, кроме Калужско-Рижской линии, были возвращены к старым записям вскоре примерно в середине 20-х чисел мая.

На новый, 2020 год в новогодних поездах пассажиров поздравляли Александр Розембаум, Рената Литвинова, Леонид Агутин, Баста, Вера Брежнева, Олег Газманов, Анжелика Варум, Владимир Кристовский, Надежда Бабкина, Сергей Мазаев, Владимир Пресняков, Антон Беляев, Дмитрий Маликов и Юлия Зиверт.

Подвижной состав
Электроподвижной состав
Основу парка подвижного состава Московского метрополитена составляют электропоезда, получающие питание от контактного рельса. Большинство электропоездов используется для пассажирских перевозок, отдельные вагоны модернизируются для использования в служебных целях и ходят в составе служебных поездов. Нумерация в метрополитене осуществляется только повагонно без присвоения номеров составам, при этом тип вагона, в отличие от его номера, не маркируется на его корпусе.
Весь электроподвижной состав Московского метрополитена метрополитена, за исключением трофейных немецких вагонов типа В, изготовлен тремя отечественными предприятиями — Мытищинским машиностроительным заводом (с 1992 г. — АО «Метровагонмаш»), ЛВЗ имени И. Е. Егорова (в 1992—2013 гг. — ЗАО «Вагонмаш») и Октябрьским электровагоноремонтным заводом.
ЛВЗ имени И. Е. Егорова до 1995 года поставлял в Москву вагоны Ем-508/509, Ем-508Т, 81-720.1, 81-580 и часть 81-717/714. Октябрьский электровагоноремонтный завод с 2018 года поставлял в Москву часть вагонов модели «Москва» и «Москва-2020». Все остальные типы вагонов в Московский метрополитен со дня его основания поставлял Мытищинский машиностроительный завод.
В соответствии с расчётами Октябрьского электровагоноремонтного завода, средняя стоимость вагона метро в период с 2010 по 2014 год варьировалась от 19 млн до 25 млн руб., что не соответствовало расчётам руководства Московского метро, которое в Арбитражном суде в 2018 году настаивало, что вагоны метро, пострадавшие в катастрофе в Московском метрополитене в 2014 году, приобретались у производителя по цене от 57 до 64 млн рублей за штуку. Суд счёл данные утверждения как недостоверные.
Цена вагона «Москва» на 2016 год составляла 65 млн рублей за штуку с расчётным сроком службы не менее 30 лет.
В Московском метрополитене используются электропоезда пяти поколений; с 2010 года идёт деятельное обновление парка подвижного состава, в частности — старейшие и отслужившие свой срок списываются, менее старые проходят капитально-восстановительный ремонт с целью продления этого срока, а также интенсивно закупаются новые.
По состоянию на июнь 2022 года вагоны нового поколения составляли 70 % от их общего количества («Русич» — 11,5 % от общего парка, «Ока» — 21 %, «Москва» — 25 %, «Москва-2020» — 12,6 %).
За последние 10 лет полностью обновлён подвижной состав на Филёвской, Арбатско-Покровской, Кольцевой, Калининской, Серпуховско-Тимирязевской, Таганско-Краснопресненской и Калужско-Рижской линиях. Инвентарный парк вагонов составляет около 6000 шт. (в среднем за сутки). Самой последней линией, на которой, в рамках обновления подвижного состава, уберут вагоны серии 81-717/714 с модификациями и 81-717.6/714.6, станет Люблинско-Дмитровская, — к 2030 году, начав обновление приблизительно в 2027 году. Люблинско-Дмитровская линия является единственной в Московском метрополитене, на которой эксплуатируется модель вагонов 81-717.6/714.6.
Таганско-Краснопресненская линия является первой, на которой эксплуатировалось сразу 4 разных вида подвижного состава. В дальнейшем таковой стала Калужско-Рижская до отстранения с неё вагонов серии 81-717/714.

Электропоезда, снятые с пассажирской эксплуатации
Электропоезда, находящиеся в эксплуатации
15 мая 2023 года Максим Ликсутов объявил, что в Москве ведётся разработка «Поезда 2030».

Именные поезда
С 1984 года в Московском метрополитене существует традиция создания именных поездов — составов, имеющих собственные имена, присвоенные в честь каких-либо событий, юбилеев, или в рамках тематических акций. Такие составы имеют различные отличительные особенности: от простых табличек и надписей с названием поезда на головных вагонах до полностью оригинального оформления всех вагонов или даже конструктивных отличий вагонов состава от обычных вагонов того же типа. В 1991 году почти все именные поезда (за исключением одного) прекратили своё существование, по причине того что их тематика носила политический характер. В начале 2000-х годов традиция была возобновлена и уже к концу 2010 года по линиям курсировало 7 таких составов различной тематики. С 2015 года процесс становится наиболее популярным, однако к настоящему времени фактически утратил своё значение. Помимо пассажирских, в Московском метрополитене имеются также два служебных именных поезда.

Диагностические поезда
В Московском метрополитене эксплуатируется несколько диагностических поездов для проверки состояния пути и выявления его повреждений с целью их последующего устранения. В составе таких поездов обычно имеется не менее пяти вагонов, включая один или реже два специальных вагона-лаборатории с измерительной аппаратурой (путеизмерители или дефектоскопы), переоборудованных из пассажирских, и несколько стандартных вагонов сопровождения. Салон вагонов-лабораторий полностью переоснащается, снаружи устанавливаются измерительные датчики, с тележек снимаются тяговые двигатели. Обычно такие вагоны имеют двухцветную жёлто-красную окраску (жёлтый верх и красный низ). Поезда выезжают на линии в дневное или вечернее непиковое время и могут обследовать несколько разных линий.
В основном в качестве диагностических использовались поезда из вагонов типов Г (до 1985 г.), Еж3, а позднее 81-717/714. Самыми старыми вагонами-путеизмерителями, долгое время эксплуатировавшимися в составах из более новых вагонов, были вагоны А № 1031 (переоборудован в музейный пассажирский) и УМ-5 № 806 (сохранён как музейный путеизмеритель). В настоящее время в качестве диагностических используются поезда из вагонов 81-717/714, включая два самых новых и технологически оснащённых диагностических комплекса «Синерги́я-1» и «Синерги́я-2», которые целиком окрашены по жёлто-красной схеме и вагон-путеизмеритель модели 81-714 № 7374, а также вагон-лаборатория СЦБ модели Еж3 № 5564.

Грузовые поезда
Ранее в Московском метрополитене эксплуатировались электропоезда для служебно-грузовых перевозок, которые с 2009 года стали применяться исключительно в роли тяговых единиц для перегонки между депо других вагонов. Грузовые поезда формировались из трёх обычных пассажирских электровагонов, один из которых переоборудовался в депо для перевозки тяжёлых и габаритных грузов.
В разное время в Москве эксплуатировались грузовые поезда из вагонов типа Д, Е/Еж и 81-717/714. Поезда ходили в дневное и вечернее непиковое время. До сентября 2009 года в Московском метрополитене эксплуатировалось 9 грузовых поездов (5 типов Е/Ем-508/Ем-509/Еж и 4 типа 81-717/714), но с декабря 2009 года грузовые поезда перестали использоваться по прямому назначению после инцидента с одним из них на Таганско-Краснопресненской линии: перевозимая в вагоне типа «Е» № 3361 колёсная пара упала с транспортёра и выдавила наружу дверь вагона с выходом её из габарита на станции «Сходненская». К настоящему времени все грузовые поезда отстранены от эксплуатации.

Контактно-аккумуляторные электровозы
Во второй половине XX века в Московском метрополитене наряду с мотовозами применялись контактно-аккумуляторные электровозы для перегонки вагонов и вождения служебных поездов в ночное время при отключённом напряжении на контактном рельсе либо на неэлектрифицированных участках метрополитена, а также при манёврах на территории депо (скорость при движении на аккумуляторах была сильно ограничена). Также они применялись на спецветках. Конструктивно контактно-аккумуляторные электровозы основывались на обычных пассажирских электровагонах, но отличались от них наличием двух кабин управления по концам вагона и размещением в пространстве салона аккумуляторных батарей вместо пассажирских сидений. По состоянию на конец 2000-х годов все контактно-аккумуляторные электровозы были выведены из эксплуатации.

Подвижной состав с двигателями внутреннего сгорания
Наряду с электропоездами в метрополитене также имеется значительный парк мотовозов, автомотрис и специализированной путевой техники для обслуживания инфраструктуры, хозяйственных грузоперевозок и доставки бригад рабочих в ночное время, при отключённом напряжении, с бензиновыми и дизельными двигателями. Значительная часть используемой техники унифицирована с применяемой на обычных железных дорогах, часть разработана специально для эксплуатации в метрополитене. Некоторые мотовозы могут применяться как в качестве локомотивов-тягачей, так и в качестве автомотрис благодаря наличию площадок для перевозки груза или сидячими местами для доставки бригад рабочих; часть автомотрис предназначена исключительно для перевозки пассажиров (рабочих).
Эксплуатируются следующие модели подвижного состава:

мотовозы и грузо-пассажирские автодрезины: МК2/15, АГМ (АГМу, АГМс), ДМ (ДМм, ДМс), АЛг, ТГК2, АГМС, МГМ1, 81-730.05, МТ, ТМ, МТК-1;
пассажирские автомотрисы: АС1, АС1А, СМДК-Мтр, РА1 730.15, автомотриса на базе вагона «Ока», ранее эксплуатировался четырёхвагонный дизель-поезд ДПС;
самоходные краны: КМ, КМП;
выправочные подбивочно-рихтовочные машины: ВПРС-02, ВПРС-500;
снегоуборочные машины: СММ2.

Электродепо
Действующие
Московский метрополитен в настоящее время обслуживается 23 электродепо (в перспективе — 31).

Три электродепо — «Сокол», «Печатники» и «Солнцево» — имеют действующие гейты с железной дорогой.

Строящиеся
После 2025 года планируется ввод в эксплуатацию:

Эскалаторы
Всего на станциях и в переходах Московского метрополитена установлены 1032 эскалатора и 2 траволатора (11 из них временно законсервированы), в том числе 18 на станциях Монорельсовой транспортной системы.
Самые длинные эскалаторы расположены на станциях «Парк Победы» (около 130 метров) и Марьина Роща (130), а два самых коротких — в южном вестибюле станции «Саларьево» (3,1). Кроме «Парка Победы» ещё на двух станциях есть эскалаторы с высотой подъёма более 60 метров — «Петровско-Разумовской» (61,2 м) и «Тимирязевской» (60,4 м). На станции «Аэропорт» и в совмещённом вестибюле станций «Чеховская» и «Пушкинская» высота подъёма эскалаторов составляет 3,2 м.

Безопасность
Объекты инфраструктуры метрополитена законодательно отнесены к особо опасным, что обусловлено большим пассажиропотоком и сложной системой инженерных сооружений. Для обеспечения технической безопасности существуют специализированные подразделения метро, численностью около 1000 человек. Они занимаются ликвидацией последствий чрезвычайных ситуаций, устранением технических неисправностей и содержанием технических средств метро. Периодически проводятся учения по координации взаимодействия работников метрополитена и экстренных служб города; ежегодно в метро гибнет не менее ста человек. Например, в 2009 году в Московском метро погибло более 150 человек и 1568 было травмировано. В 2023 году падения пассажиров на пути в Московском метрополитене происходят несколько раз в неделю, иногда — дважды в день.
Охраной правопорядка занимается Управление внутренних дел на Московском метрополитене. Его штат составляет более 5 тысяч человек. Основные направления деятельности управления: проверка мигрантов, борьба с вандализмом, хулиганством, кражами и грабежами. Большое внимание уделяется противодействию терроризму.
Тем не менее, были проблемы и в этой сфере. Например, по данным учёных Института проблем экологии и эволюции имени А. Н. Северцова РАН на 2008 год, в Московском метрополитене обитало около пяти сотен бездомных собак. Некоторые из них целенаправленно используют метро в качестве транспорта.
В Московском метрополитене имеются различные технические средства контроля. На многих станциях и вагонах установлены системы видеонаблюдения. Телевизионные камеры, установленные в вагоне, записывают и сохраняют в архиве кадры того, что происходит в вагоне. Данная видеоинформация в дальнейшем может оказать помощь в изучении обстоятельств произошедших событий. У видеокамер, установленных на станциях, есть ещё одна функция — управление работой станций. Также в последнее время на станциях и платформах устанавливаются камеры с системой интеллектуального видеонаблюдения с функциями видеораспознавания, видеообнаружения и видеомониторинга. Распоряжением Правительства РФ от 4 июля 2019 года № 1460-р, опубликованным на официальном интернет-портале правовой информации, на оснащение техническими средствами обеспечения транспортной безопасности государство выделит 263 млн рублей, зато сама столица направит на эти цели 5 млрд рублей.
На всех станциях Московского метрополитена установлены специальные колонны экстренного вызова, которые предназначены для связи пассажиров с Ситуационным центром. С их помощью также можно получить справочную информацию по работе метрополитена у оператора, а также сообщить об экстренной ситуации. Кроме того, пассажир может обратиться к дежурному по станции (дежурные носят форму с красным головным убором).
В салоне вагонов есть устройства связи с машинистом поезда. В вагонах, где установлены видеокамеры централизованной системы видеонаблюдения, устройства связи переключены на Ситуационный центр. При поступлении вызова с такого устройства связи, к вызвавшему вагону подключится оператор Ситуационного центра. Оператор, помимо общения через устройство, может наблюдать видеоизображение из вагона, передаваемое одновременно с нескольких камер (видимых и скрытых), визуально оценивая происходящее.
В марте 2016 года руководитель Московского метрополитена Дмитрий Пегов объявил, что в будущем пассажиры подземного транспорта будут оповещены в случае сбоев в его работе и получат SMS. Данный сервис уже успел пройти апробацию, когда в начале марта 2016 года Калужско-Рижская линия работала с ограничениями. Развивать сервис информирования пользователей подземки планируется совместно с компанией «МаксимаТелеком».

Служба безопасности Московского метрополитена
Деятельность Службы безопасности Московского метрополитена регламентируется:

Федеральным законом «О транспортной безопасности» от 09.02.2007 № 16-ФЗ;
Постановлением Правительства Российской Федерации от 5 апреля 2017 г. N 410 «Об утверждении требований по обеспечению транспортной безопасности, в том числе требований к антитеррористической защищённости объектов (территорий), учитывающих уровни безопасности для различных категорий метрополитенов».
Лица, отказавшиеся от досмотра, на территорию метрополитена не допускаются. Отказ пассажира от досмотра, дополнительного досмотра и повторного досмотра является основанием для расторжения договора перевозки в одностороннем порядке. Попытка пройти без досмотра является административным правонарушением и может повлечь за собой наложение штрафа.

Московский метрополитен как объект гражданской обороны
Почти с первых лет существования Московского метрополитена предполагалось использовать его в качестве объекта гражданской обороны. В апреле 1941 года вышло постановление Совнаркома, согласно которому метрополитен должен был быть приспособлен под массовое бомбоубежище. Во время Великой Отечественной войны здесь скрывались от авианалётов тысячи москвичей.
После войны новые станции проектируются с учётом возможного использования потенциальным противником оружия массового поражения: атомного, химического и бактериологического. Вентиляционные шахты оборудуются фильтрами. На перегонах строятся санузлы, рассчитанные на обслуживание большого количества людей (в мирное время не используются).
В тоннелях, а также на выходах со станций метро сооружаются герметичные двери (гермозатворы). Они могут выдержать воздействие взрывной волны, а также способны предотвратить затопление станций и тоннелей в случае наводнения. Но гермозатворы не обеспечивают абсолютной изоляции. На случай отключения городского электропитания предусмотрены дизельные электростанции. Их мощности достаточно для поддержания освещения и вентиляции.

Наряду со служебными объектами самого метрополитена (фильтро-вентиляционные камеры, командные пункты воздухозабора и воздуховыпуски, дизельные электростанции, залы с аппаратурой связи и т. д.), с его инфраструктурой интегрированы многие правительственные и ведомственные бункеры глубокого заложения, которые выполняют роль защищённых рабочих помещений и резервных командных пунктов. Данные объекты как правило расположены на одном уровне с перегонными тоннелями метрополитена и имеют с ними сбойки, а в некоторых случаях и проходы на станции. Это позволяет в условиях чрезвычайных ситуаций эвакуировать их персонал и осуществлять их снабжение и резервную вентиляцию из тоннелей, а в обычное время метрополитен может использоваться для перевозки на эти объекты хозяйственных грузов. Большинство из этих бункеров было построено в советский период метростроевцами совместно с объектами метрополитена, хотя их расположение, конфигурация и даже сам факт существования были строго засекречены не только для граждан, но и для большинства сотрудников метрополитена. По состоянию на 2020-е годы, большая часть бункеров, построенных до 1960-х годов, рассекречена, при этом многие из них более не используются из-за технического и морального устаревания, однако для посещения открыты только два из них, бункер-42 и бункер-703, которые превращены в музеи (при этом проход в тоннели метрополитена для посетителей из них закрыт); остальные объекты продолжают выполнять свою функцию и многие из них по-прежнему засекречены.
В XXI веке Московский метрополитен отказался от оборудования новых станций с возможностью использования их как убежищ.

Аварии и теракты в Московском метрополитене
Сергей Собянин считает Московский метрополитен одной из самых надёжных систем метро мира, этой же точки зрения придерживался и бывший начальник ГУП «Московский метрополитен» Дмитрий Гаев. Однако в истории метрополитена тоже случались происшествия с человеческими жертвами. Первый теракт в истории московского метро произошёл 8 января 1977 года. В вагоне, ехавшем на открытом участке перегона между станциями «Измайловская» и «Первомайская», сработало взрывное устройство, что привело к гибели людей. По приговору суда, проходившего в обстановке строжайшей секретности, обвиняемые были расстреляны.
Крупная авария случилась 17 февраля 1982 года на станции метро «Авиамоторная». В вечерний час пик произошла поломка эскалатора, ведущего на спуск. В результате под тяжестью пассажиров эскалатор развил скорость, в 2—2,4 раза превышающую номинальную. На сходе с эскалатора люди не могли удержаться на ногах, и поэтому случилась давка, в которой погибло 8 человек.
Второй теракт произошёл 11 июня 1996 года. В вагоне поезда, следовавшего со станции «Тульская» на станцию «Нагатинская», сработало самодельное взрывное устройство. В результате взрыва погибли 4 человека. Это преступление до сих пор остаётся нераскрытым.
Следующий теракт, повлёкший гибель людей, произошёл 8 августа 2000 года в подземном переходе станции «Пушкинская». Жертвами взрыва бомбы стали 13 человек.
6 февраля 2004 года в вагоне поезда, следовавшего в центр между станциями «Автозаводская» и «Павелецкая», прогремел взрыв. Бомбу привёл в действие террорист-смертник. В результате взрыва погиб 41 человек.
31 августа 2004 года произошёл ещё один теракт. Террористка-смертница взорвала себя у вестибюля станции метро «Рижская». Теракт унёс жизни 10 человек.
28 апреля 2005 года на станции «Выхино» под прибывающий поезд бросилась 22-летняя девушка. Машинист успел применить экстренное торможение, благодаря чему она отделалась лёгкими травмами, однако 47-летний машинист через полчаса скончался от сердечного приступа, не выдержав пережитого.
Серия террористических актов произошла в московском метро утром 29 марта 2010 года. Первый взрыв прогремел в 7:57 в вагоне поезда на станции метро «Лубянка»; второй — в 8:37 на станции «Парк культуры» Сокольнической линии. Оба теракта были осуществлены при участии террористок-смертниц. Жертвами взрывов стали 40 человек.
15 июля 2014 года в 8:39 утра произошёл сход с рельсов состава, следовавшего по Арбатско-Покровской линии от станции «Парк Победы» до станции «Славянский бульвар». По данным Следственного комитета, причиной катастрофы стала неисправность стрелочного механизма. По данным на утро 16 июля, число пострадавших составило не менее 160 человек. Более 100 из них госпитализированы. 24 человека погибли.

Бродячие собаки
Начиная с конца 1990-х, когда в городе проводилась программа ОСВВ, предусматривавшая свободное нахождение стерилизованных бродячих собак в городской среде, на станциях метро и в подуличных переходах жили эти животные. Это приводило к конфликтам. В частности, в 2001 году на станции «Менделеевская» владелица породистого пса убила напавшего на её животное бродячего кобеля по кличке Мальчик, что вызвало широкий общественный резонанс. Позднее на станции «Охотный Ряд» неизвестный столкнул под колёса прибывающего поезда бродячую собаку, в результате чего та повредила позвоночник и осталась инвалидом. По состоянию на 2008 год, по данным Института экологии и эволюции РАН, на объектах метрополитена ежедневно находились до 500 собак, которые ездили в вагонах, лаяли на пассажиров, пугали их, либо выпрашивали еду. В 2009 году, после того как городские власти Москвы остановили программу ОСВВ и возобновили прекращённый в конце 1990-х гуманный безвозвратный отлов собак с помещением их в муниципальные приюты, безнадзорные звери с транспортных объектов стали изыматься. Так, в частности, поступили со случайно попавшей в метро бродячей собакой и её щенками, родившимися в вагоне Кольцевой линии на глазах у пассажиров в 2016 году.

Дискуссия об установке платформенных заграждений
На начало 2025 года на станциях Московского метрополитена платформенные раздвижные двери отсутствуют. Существовали планы по их установке на Каховской и Бутовской линиях, на станциях Большой кольцевой линии «Хорошёвская», «Петровский парк», «Савёловская» и «ЦСКА». В 2008 году Генпрокуратура даже требовала, чтобы раздвижные перегородки были установлены на всех станциях, однако затем от этих планов было решено отказаться, так как использование платформенных раздвижных дверей может привести к увеличению интервалов движения поездов в условиях высокого пассажиропотока.

Рекорды (особенности и статистика)
Станции
Самая загруженная станция по состоянию на III квартал 2025 года — «ВДНХ» (146 тыс. человек в сутки).
Самый загруженный пересадочный узел — «Комсомольская»—«Комсомольская» (206 тыс. человек в сутки).
Наименее загруженная станция (не считая пересадочных) — «Тютчевская» (6,8 тысяч человек в день).
Самая глубокая станция — «Парк Победы» (73 м).
Наименее глубокая станция — «Печатники» Люблинско-Дмитровской линии (5 м).
Станции, находящиеся только наполовину под землёй, — «Мичуринский проспект» и «Пыхтино», обе на Солнцевской линии.
Самая длинная станция — «Воробьёвы горы» (282 м), она же является единственной станцией в Московском метрополитене и одной из двух на постсоветском пространстве, расположенной на метромосте (вторая — «Аметьево» Казанского метрополитена).
Самая «кривая» станция — «Александровский сад» (радиус кривизны платформы — 750 м), она же имеет самый продолжительный статус конечной станции (с 7 ноября 1958 года — после повторного открытия в составе Филёвской линии).
Самая узкая станция — «Москва-Сити» Филёвской линии (от края одной платформы до края другой — 11,8 м, ширина каждой из платформ — около 2 м).
Самая широкая станция — «Нижегородская» (58 м), она же является единственной пятипролётной колонной станцией в Московском метрополитене и единственной станцией с четырьмя путями.
Станция с самым коротким сроком в качестве конечной станции — «Румянцево», обладавшая статусом конечной 4 недели (с 18 января по 15 февраля 2016 года) до открытия станции «Саларьево».
Самая «тихая» станция — «Пражская», за счёт облицовки её путевых стен особым видом керамической плитки, содержащей в себе воздушные поры, поглощающие звук.
Станция «Тверская» — первая в мире, открытая на действующем перегоне («Маяковская» — «Театральная») без остановки движения поездов и строительства обходных тоннелей.
Станция «Котельники» — единственная станция в Московском метрополитене, выходы из которой ведут сразу в три разных населённых пункта: в Москву (район Выхино-Жулебино), а также в города Котельники и Люберцы Московской области.
Станция с самым длительным сроком строительства — «Спартак» (1975—2014, то есть 39 лет с учётом консервации).
Станции, претерпевшие наибольшие количество переименований — «Александровский сад», «Охотный Ряд» и «Партизанская» (все — 4 раза).
Самый длинный перегон между станциями в московском метро, доступными для пассажиров — «Крылатское» — «Строгино» (6625 м).
Самый короткий перегон в Московском метрополитене — «Деловой центр» — «Москва-Сити» (497 м).
Станция «Мякинино» — первая и единственная в российской практике метростроения, построенная с участием частных инвестиций — компании «Crocus Group» Араза Агаларова, а также первая и пока что единственная станция, расположенная целиком на территории Московской области.
Станция «Потапово» — первая и на данный момент единственная отапливаемая наземная крытая станция в России.

Линии
В апреле 2025 года Департамент транспорта Москвы опубликовал данные, согласно которым Калининская (47,8 км/ч), Некрасовская (45,8 км/ч) и Троицкая (45,2 км/ч) являются линиями с самой большой средней скоростью движения поездов.
Большую кольцевую линию (до начала обслуживания Сокольнической линии электродепо ТЧ-23 «Столбово», помимо ТЧ-1 «Северное» и ТЧ-13 «Черкизово») обслуживают три электродепо: ТЧ-7 «Замоскворецкое», ТЧ-21 «Нижегородское» и ТЧ-22 «Аминьевское». Кроме того, Большая кольцевая линия является самой длинной кольцевой линией в мире с длиной 57,5 км, однако к 2030 году уступит этот статус линии 15 Парижского метрополитена — 75 км, в том числе включая служебные соединительные ветви и подъездные (парковые) пути к ангарам в электродепо.
В честь первоначального названия Замоскворецкой линии (Горьковский радиус) назван вагон метрополитена типа «Г» (81-701).

Иные объекты
Самый длинный метромост — Лужнецкий (2030 м).
Самый длинный эскалатор находится на станции «Марьина Роща» Большой кольцевой линии (130 м).
Самый глубокий тоннель — «ветка» Метро-2 — имеет глубину 189 м в районе спецобъекта № 54, расположенного возле станции «Университет». Хотя формально система Метро-2 и часть соединительного тоннеля за гермоворотами после передачи на баланс КГБ не относится к Московскому метрополитену начиная с 1966 года, на момент её постройки этот тоннель можно считать самым глубоким.

Прочее
Календарный год, в котором введено в строй наибольшее за всю историю количество станций, — 2018 год (17 станций).
День с наибольшим пассажиропотоком за всю историю метрополитена — 6 сентября 1997 года (14 миллионов человек). Рекордный пассажиропоток был обусловлен празднованием 850-летия Москвы.

Расположение
Московский метрополитен — второй в России после Петербургского (со ст. «Девяткино» в г. Мурино) с расположением станций как в административно-территориальных границах города, так и за ними — в области, охватывая агломерацию: так, в области находятся станции «Мякинино» и «Котельники» (из них «Мякинино», ввиду финансирования правительством области и частными капиталовложениями — компанией Crocus Group Араза Агаларова, не эксплуатируется ГУП «Московский метрополитен» и не принадлежит правительству Москвы). Станция «Мякинино» находится в Красногорске, а «Котельники» — в г. Котельники. Выходы с «Котельников» также ведут в район Москвы Выхино-Жулебино (ЮВАО) и г. Люберцы.
Также в планах строительства после 2035 года — продление Калужско-Рижской линии от станции «Медведково» в сторону деревни Челобитьево в городе Мытищи с одноимённой станцией и промежуточной «Челобитьево» и Таганско-Краснопресненской в деревню Куркино.
На территории области в том числе находятся выходы со станции «Новокосино» — они ведут в г. Реутов.

Перспективы
Строящиеся и планируемые станции и линии
К 2032 году планируется продлить Сокольническую линию за станцию «Бульвар Рокоссовского» до Ярославского шоссе, построив две новые станции — «МГСУ» и «Ярославская», перегон между которыми будет аналогичен перегону, находящемуся между станциями «Крылатское» и «Строгино» на Арбатско-Покровской линии с эвакуационным выходом в виде технической платформы «Троице-Лыково» с предполагаемой длиной 5 км, требующейся для протяжённых участков пути между станциями: перегон «Крылатское» — «Строгино» является самым длинным в московском метро — 6,6 км, время поездки по нему составляет 7 минут.
После 2035 года, согласно новой перспективе развития, планируется продление Калужско-Рижской линии до Мытищ с одноимённой станцией и промежуточной «Челобитьево» после «Медведково», соединение Калининской и Солнцевской линий в единую Калининско-Солнцевскую (одновременно с этим отсутствующее в утверждённой программе строительства) и продление Таганско-Краснопресненской линии до Куркино.

Модернизация инфраструктуры
В перспективах развития также переход к модернизации освещения тоннелей. Новые светильники обещают быть устойчивее к вибрациям, вызываемым движением поездов, потребляют меньше энергии, чем их предшественники, а также имеют увеличенный срок годности. Все светильники будут подключены к одной системе и могут управляться удалённо.

Беспилотное движение
В Московском метрополитене существуют планы по запуску движения беспилотных поездов. Ещё в 2011 году планировалось организовать движение поездов без машинистов на Бутовской линии. В 2023 году провести тестирование системы автоведения поездов с сохранёнными кабинами машинистов планировалось на Кольцевой линии (такие поезда, но без кабин машинистов, распространены в метрополитенах, станции которых оснащены платформенными раздвижными дверьми). В том же 2023 году мэр Сергей Собянин сообщил о планах запуска беспилотных поездов к 2026 году. Однако переходу к беспилотному движению препятствует человеческий фактор — психологическая неподготовленность людей к факту управления поезда роботом.

Автоматическое ведение
В январе 2026 года был представлен первый состав из вагонов 81-775.2/776.2/777.2 «Москва-2024», оборудованный системами автоведения в тестовом режиме. Его планируется внедрить к 2030 году. Различные версии систем автоведения на более старом подвижном составе (в частности вагонах «Д», Еж3/Ем-508Т и 81-717/714) ранее уже внедрялись на Калужско-Рижскую, Ждановско-Краснопресненескую и Калининскую линии; испытания систем проводились в 1960-х годах.

Альтернативные проекты развития
До принятия текущей программы развития московского метрополитена существовали альтернативные проекты развития линий, наиболее известными из которых являлись:

Большое кольцо Московского метрополитена — проект, который был задуман ещё в плане развития метрополитена в 1960-х годах.
Хордовые линии Московского метрополитена — ряд проектов, которые начали разрабатываться в 1980-х годах.

Проблемы при реализации планов
Основными минусами Программы развития Московского метрополитена являются регулярно срываемые сроки сдачи объектов и растущие по отношению к запланированным затраты на строительство. Нынешняя программа развития была принята в 2011 году, и согласно ей к 2020 году в Москве должно было быть открыто 78 новых станций и построено 160 км линий. Заявлялось, что темпы строительства будут превышать прежние в два раза, а затраты на возведение метро за счёт экономии снизятся на 25—30 % по сравнению с лужковским периодом.
В начале каждого года мэрия регулярно обещает сдать большой объём тоннелей и станций в наступившем году, каждый раз больше, чем в предыдущем. Реальные же темпы строительства метро как минимум вдвое ниже заявленных. За 6 лет с 2011 по 2017 год построено 49,7 км новых линий и 25 станций из обещанных 124 км и 58 станций. В январе 2017 года руководитель Департамента строительства города Москвы Андрей Бочкарёв неоднократно заявлял, что в 2017 году будет сдано 19 новых станций. Указанные обещания были реализованы полностью с полугодовой-годовой задержкой.
Также неисполненными остались обещания о сокращении трат на проектирование и строительство метро на 30 %. По программе «Развитие транспортной системы Москвы», в 2012—2016 годах на строительство объектов метро планировалось направить 465,6 млрд, а в целом до 2020 года — 900 млрд рублей. В программе, откорректированной осенью 2014 года, инвестиции выросли до 1,08 трлн руб. По факту на конец 2016 года потрачено 629,5 млрд рублей, и согласно новой Адресной инвестиционной программе Москвы за 2017—2020 годы на развитие метрополитена планируется дополнительно потратить ещё 759,4 млрд рублей. Таким образом, сумма трат превысит 1,38 трлн рублей, что уже на 25 % больше первоначально запланированных вложений.

Экскурсии в метро
«Метротур» — проект Городского экскурсионного бюро и Московского метрополитена. Гиды проходят трёхдневное обучение: получают информацию об истории развития метро, обмениваются опытом проведения экскурсий, знакомятся с правилами пользования и эксплуатации метрополитена, техникой безопасности и порядком работы в случае чрезвычайных происшествий.
Программа «Метротура» включает в себя около 15 экскурсий, среди которых содержатся как экскурсии по публично доступным станциям (экскурсии «Московская подземка. Краткий курс», «Как мы строили метро», «Большой сталинский стиль в московском метро», «Жемчужины Замоскворецкой линии», «Семь красок московского метро», «Путешествие в царство света» и другие), так и по некоторым электродепо, на территорию которых доступ для гражданских лиц вне экскурсий закрыт (экскурсии «Как устроено метро»).
В мае 2016 года в рамках акции «Ночь в музее» состоялась первая ночная экскурсия. Пассажиры проехали по первой линии Московского метрополитена — Сокольнической на ретропоезде «Сокольники», стилизованном под состав типа А 1935 года. Маршрут экскурсии проходил от станции «Сокольники» до «Парка культуры» со всеми остановками. Позднее, после восстановления ретропоезда из трёх настоящих вагонов типа «А» до ходового состояния, ночные экскурсии стали проводиться на нём.

«Книги в метро»
Проект «Книги в метро» был запущен летом 2018 года. Впервые виртуальные книжные полки стали доступны на выполненной в виде библиотеки станции «Рассказовка». В электронной библиотеке метрополитена собрано более 100 произведений классиков русской и мировой литературы. Весной 2019 года онлайн-библиотека столичной подземки пополнилась 600 новыми произведениями.
Проект был перезапущен в мае 2023 года. На первом этапе пассажирам стали доступны 417 произведений русской и мировой художественной литературы, которые можно было бесплатно читать из приложения «НЭБ Свет» от Российской государственной библиотеки. Перезапуск прошёл в рамках масштабного спецпроекта «Читаем в дороге», к которому также присоединились «Аэрофлот» и сервис каршеринга «BelkaCar». В сентябре сообщалось, что читателям доступны более 3 тысяч произведений, а QR-коды с художественной литературой доступны на стойках «Живое общение». Самыми популярными произведениями у пассажиров на тот момент стали «Приключения Тома Сойера» Марка Твена и «Антоновские яблоки» Ивана Бунина.

Московский метрополитен в культуре
Литература
Подготовка к строительству метрополитена в Москве стала темой вышедшего в 1933 году романа В. В. Варанкина «Metropoliteno», который литературоведы считают одним из классических произведений эсперантской прозы.
Строительству первой очереди Московского метрополитена посвящены очерки советских писателей И. Ильфа и Е. Петрова, чешского журналиста Юлиуса Фучика и других авторов. На фоне строительства первой очереди разворачивается действие и повести С. П. Антонова «Васька» (1987).
В современной русской литературе известен ряд художественных произведений, которые посвящены событиям, происходящим в московском метро. Например, в постапокалиптических романах Дмитрия Глуховского «Метро 2033», «Метро 2034» и «Метро 2035» описывается жизнь людей в московском метро после ядерной войны. Помимо романов Глуховского, постапокалиптическое московское метро описано также в романах книжной серии «Вселенная Метро 2033»:

Владимира Березина «Путевые знаки»;
Сергея Антонова «Тёмные туннели», «В интересах революции», «Непогребённые», «Рублёвка», «Рублёвка-2. Остров блаженных», «Рублёвка-3. Книга Мёртвых», «Харам Бурум»;
Андрея Ерпылева «Выход силой»;
Сергея Кузнецова «Мраморный рай»;
Сурена Цормундяна «Странник»;
Анны Калинкиной «Станция-призрак», «Царство крыс», «Кошки-мышки», «Хозяин Яузы», «Обмануть судьбу», «Спастись от себя», «Сетунь»;
Сергея Зайцева «Санитары», «Тёмная мишень»;
в нескольких рассказах сборников «Последнее убежище», «Сумрак в конце туннеля», «Сказки апокалипсиса» «О чём молчат выжившие», «Холодное пламя жизни» и «Они не те, кем кажутся»;
Захара Петрова «Муос»;
Тимофея Калашникова «Изнанка мира»;
Андрея Гребенщикова «Обитель снов», «Сёстры печали»;
Ольги Швецовой «Стоящий у двери», «Ничей», «Демон-хранитель»;
Виктора Лебедева «Рождённые ползать», «Летящий вдаль»;
Татьяны Живовой, Алексея Матвеичева и Павла Гаврилова «Джульетта без имени», а также отдельно Татьяны Живовой «Пасынки Третьего Рима»;
Евгения Шкиля «Гонка по кругу»;
Марии Стреловой «Изоляция»;
Павла Макарова «Перекрёстки судьбы»;
Сергея Москвина «Пифия», «Пифия-2. В грязи и крови»;
Станислава Богомолова «Нити Ариадны»;
Светланы Кузнецовой «Уроборос», «Дворец для рабов»;
Андрея Лисьева «Зима милосердия».
В книге Дмитрия Сафонова «Метро» описывается вымышленная трагедия, произошедшая с составом, следовавшим от станции «Тушинская» до станции «Щукинская». В произведении также упоминаются реально существующие люди, в том числе бывший начальник Московского метрополитена Дмитрий Гаев (в книге — Игорь Маев).
В сериях книг «Пешка в большой игре» и «Рок-н-ролл под Кремлём» российского писателя Данила Корецкого автор уделяет в сюжете много внимания теме московских подземных коммуникаций в целом и метрополитену в частности. Писатель отыгрывает «городские легенды» о крысах-мутантах и пауках-мутантах и вводит свои собственные — об одичавших за столетия низкорослых людях, целые племена которых населяют подземные пустоты столицы.
Загадочному и таинственному миру московского метрополитена, его мистической связи с религиозным наследием античности посвящено философское эссе Виктора Пелевина «Подземное небо».
Работа метрополитена глазами машиниста метропоезда описана в книге Олега Дивова и Максима Рублёва «Не прислоняться (правда о метро)».

Музыка
Московскому метрополитену посвящены многие песни. Среди них «Песня старого извозчика» (Л. Утёсов), «Песенка о московском метро» (Б. Окуджава), «Песенка о метро» (М. Миронова), «Площадь Ногина» (группа «Ковчег»), «42 минуты» (В. Сюткин, клип на эту песню снят в московском метро), «Станция „Таганская“» (группа «Любэ»), «Звёзды не ездят в метро» (группа «Машина времени», альбом «Место, где свет» (2001); позже исполнялась группой «Бумбокс»), «Случай в метро» (рок-группа «Центр»), «Метро» (группа «Високосный год»), «Метро» (Кибуц), «В метро» (Земфира), «Слёзы подземки» (О взрывах в метро в марте 2010 г.) и «Варшавское депо» (группа «Кантемир»), «Метро» (Tracktor Bowling), «Странные танцы» (группа «Технология», клип снят в московском метро) «Столько жизни» (музыкальный проект «Ассаи»), звучавшая в эфире московского метро песня «Метро» (группа «Где моё лето») и другие, а также существует инструментальная композиция «Гимн московскому метрополитену», входящая в альбом Бориса Гребенщикова и квартета Анны Карениной «Задушевные песни», вышедший в 1994 году.
У барда Михаила Щербакова есть песня «Красные ворота», в которой упоминается атрибутика метро: радиальная линия, турникет и т. д.
У автора-исполнителя Николая Воронова есть песня «Баррикадная», в которой упоминается множество названий станций московского метрополитена. Также у российского рэп-исполнителя Noize MC есть песни «Кантемировская» и «В метро», у Slim’а есть трек про метрополитен под названием «Имени Ленина», а у Guf’а — «Metropolitan Mail», у группы «Пурген» есть песня под названием «Трагедия на Авиамоторной». Песня Данила Поминова «В метро» стала саундтреком к телесериалу «Метро», пилотные серии которого были сняты в 2012 году.
У швейцарской группы Plaistow есть инструментальная композиция «Mayakovskaya», вышедшая в альбоме «The Crow» (2010), которая была названа в честь станции Московского метро.
В ирландском городе Лимерик есть группа «Moscow Metro».

В кинематографе
Художественные фильмы
«Цирк» — художественный фильм 1936 года режиссёра Григория Александрова — первое появление Московского метрополитена в художественном кинематографе. Показаны станция Охотный Ряд и эскалатор с взволнованными пассажирами.
«Добровольцы» — художественный фильм 1958 года режиссёра Юрия Егорова, посвящённый поколению строителей первой очереди Московского метрополитена.
«Город над головой» — мини-сериал (4 серии) 1985 года режиссёров Геннадия Павлова, Сергея Сатыренко по сценарию Виктора и Иосифа Ольшанских.
«Подземный храм коммунизма» — французский документальный фильм 1991 года;
«Научная секция пилотов» — художественный фильм 1996 года режиссёра Андрея И.
«Метро» — фильм-катастрофа 2013 года по мотивам романа Дмитрия Сафонова.
Короткометражный телевизионный сериал «Метро» — в 2012 году прошёл конкурс молодых кинематографистов и выпущены пилотные серии телесериала, действие которого происходит в московском метрополитене.
На станциях Московского метрополитена происходит действие фильмов российского и зарубежного производства «Летят журавли», «Я шагаю по Москве», «Застава Ильича», «Семь нянек», «Влюблён по собственному желанию», «Москва слезам не верит», «Полицейская академия 7: Миссия в Москве», «К-19», «Подсолнухи», «Ночной дозор» и множества других.
Станцией Московского метрополитена «Площадь Революции», а также его логотипом вдохновлены декорации фильма «Гарри Поттер и Дары Смерти: Часть II».

Документальные
Просто метро. О Московском метрополитене им. В. И. Ленина, 1972, реж. В. Стрелков

Телесериалы
На станции «Полежаевская» Московского метрополитена происходят действия первой серии первого сезона российского сериала «Ангел или демон».
В Московском метрополитене происходят действия седьмой серии девятого сезона российского сериала «Склифосовский»; за Московский метрополитен выдаётся Нижегородский.

Видеоигры
В Московском метрополитене разворачиваются действия некоторых видеоигр.

Действие игры «Метро-2» (2005 год) и её сиквела «Метро-2: Смерть вождя» (2006 год) разворачивается в московском метро в 1952—1953 годах.
Видеоигра «Метро 2033» (2010 год), выпущенная в жанре шутера от первого лица. Её сюжет основан на одноимённом романе Дмитрия Глуховского.
Видеоигра «Metro: Last Light» (2013 год), выпущенная в жанре шутера от первого лица. Её сюжет и диалоги созданы также Дмитрием Глуховским.
Видеоигра «Metro Exodus» (2019 год).
Часть московского метро также представлена в видеоигре «You Are Empty» (2006 год).
Почти все линии Московского метро есть в игре «Trainz 12» (2011 год).
В дополнении (моде) «Metrostroi Subway Simulator» к игре «Garry’s Mod» в Steam есть следующие линии Московского метрополитена: небольшой участок Люблинско-Дмитровской, вся Некрасовская, вся Калининская линия, участок Калужско-Рижской линии «Новые Черёмушки» — «Октябрьская»; в разработке находится участок Замоскворецкой линии «Ховрино» — «Белорусская» с электродепо ТЧ-2 «Сокол». Вышеперечисленное функционирует и будет функционировать в виде аддонов в Steam Workshop. Разработка ведётся, как правило, «игроками-энтузиастами». В дополнении также представлен пассажирский подвижной состав метрополитена (в том числе выведенный из эксплуатации) в виде вагонов «Ам», «Д», «Е», Еж/Еж1, Еж3/Ем-508Т (до капитально-восстановительного ремонта (КВР) и с ним — Еж3РУ1/Ем-508ТРУ1), 81-717/714, «Яуза» и две разновидности поезда «Ока» — со сквозным (свободным, межвагонным) проходом и без него. Дополнение отличается от других компьютерных симуляторов поездов высокой реалистичностью с широкой кастомизацией каких-либо действий. В этом же дополнении на карте вымышленной линии присутствует техническая платформа «Троице-Лыково», расположенная на Арбатско-Покровской линии после станции «Крылатское». В этом же дополнении на карте Crossline Redux с вымышленной линией воссозданы станции «Красные Ворота» (под вымышленным названием «Проспект Суворова»), «Автозаводская» (под вымышленным названием «Нахимовская»), «Октябрьская» Калужско-Рижской линии и «Новогиреево», смоделированы и подключены к системе СЦБ с основным средством сигнализации в виде автоблокировки со светофорами и защитными участками, дополненной АЛС-АРС. Из них на Сокольнической и Калининской линиях в реальности применяется система АЛС-АРС в качестве основного средства сигнализации без автостопов и защитных участков. Также на карте Virus с также вымышленной линией воссозданы станции «Савёловская» Большой кольцевой линии и «Южная» (с несуществующим у неё путевым развитием — после следования с отклонением по стрелочному переводу далее осуществляется движение, в реальности там находится оборотный тупик по перекрёстному съезду (также называемый «глухим пересечением»)), под «Лермонтовский проспект» стилизована вымышленная станция «Хорошавино». На вымышленной линии также применена автоблокировка со светофорами и защитными участками, в реальности на Большой кольцевой линии применяется АЛС-АРС без автостопов и защитных участков.
В симуляторе для смартфонов «Subtransit Drive», вышедшем на iOS 31 декабря 2022 года, присутствует Калининская линия. Релиз на Android состоялся 1 марта 2023 года. Разработчиками также планируется внедрение в игру других линий и подвижного состава: в частности, после аварии на станции «Печатники» была анонсирована разработка электропоезда «Яуза» обеих разновидностей — с коллекторными электродвигателями и асинхронными, модификации «.1»; в игре присутствует электропоезд «Русич» оригинальной (не модернизированной) модели 81-740/741 и модификация 81-740.1/741.1. «Subtransit Drive» является продолжением дополнения «Metrostroi Subway Simulator» на игровом движке Unreal Engine 4.
В игре «Counter-Strike 2» происходят действия на вымышленной станции Московского метро «Киберспортивная».

В филателии
Награды
Орден Ленина (6 сентября 1947 года) — «за образцовую организацию работы по перевозкам населения и успешное освоение новой техники».
Юбилейный почётный знак в ознаменование 50-летия образования Союза ССР (1972) — «к 50-летию образования Союза ССР»
Памятный знак «За трудовую доблесть в девятой пятилетке» (1976) — «за успехи в девятой пятилетке метрополитен»
Орден Трудового Красного Знамени (1985) — «за успешное выполнение планов перевозки пассажиров».
Почётный знак Российской Федерации «За успехи в труде» (1 мая 2023 года) — «за достигнутые высокие показатели в производственной деятельности».
Орден «За доблестный труд» (21 мая 2025 года) — за большой вклад в развитие транспортного комплекса города Москвы и достигнутые трудовые успехи.

Реклама
С 1 января 2011 года по апрель 2015 года размещением рекламы в московском метро занималось ООО «Авто Селл». Компании пришлось покинуть метрополитен из-за финансовых трудностей, договор с ней был досрочно расторгнут летом 2015 года в связи с задолженностью перед метрополитеном более 1 миллиарда рублей.
В сентябре 2016 года рекламным оператором Московского метрополитена стало ООО «Трейд Компани», выигравшее на аукционе десятилетний контракт на эксклюзивное размещение рекламы, которое компания начала осуществлять с 1 января 2017 года. В декабре 2017 года ГУП «Московский метрополитен» расторгло договор с компанией в одностороннем порядке в связи с непредоставлением ООО «Трейд Компани» финансового обеспечения контракта на 2018 год.
По состоянию на январь 2018 года ГУП «Московский метрополитен» рассматривало возможность самостоятельной продажи рекламы в метро; новый конкурс среди рекламных операторов организовывать не планировалось.
7 сентября 2018 года Арбитражный суд Москвы признал расторжение договора с ООО «Трейд Компани» незаконным; ГУП «Московский метрополитен» намеревалось оспаривать решение суда в вышестоящей инстанции и занималось подготовкой к подаче документов в Девятый арбитражный апелляционный суд.
С 2019 года Московский метрополитен в своём твиттер-аккаунте «Metrooperativno» приступил к анонсам домашних матчей ФК «ЦСКА», «Спартак», «Динамо» и «Локомотив», концертов в Лужниках, концертном зале стадиона «Динамо», православных и мусульманских праздников, а также иных крупных зрелищных событий, требующих введения временных ограничений режима работы отдельных станций метро.

Критика
Большой проблемой в московском метрополитене является недостаточная доступность метрополитена для маломобильных групп населения. В частности, не все уже открытые старые станции были оборудованы лифтами для маломобильных пассажиров. И даже на оборудованных станциях не выполняются нормативы. Также кроме световой полосы безопасности по краям платформ отсутствует тактильное покрытие (точечные предупреждающие блоки), указывающее на края платформы незрячим, и направляющие дорожки (линейные направляющие блоки), указывающие безопасный путь незрячим к краю платформы для посадки в вагоны поездов.

См. также
Общественный транспорт Москвы
Окаменелости в Московском метрополитене
Список станций Московского метрополитена
История Московского метрополитена
Московский Метрострой

Примечания
Комментарии

Источники

Литература
Нойтатц Д. Московское метро: от первых планов до великой стройки сталинизма (1897–1935) / пер. с нем. Ю. А. Петрова. — М. : Росспэн, 2013. — 783 с. — ISBN 978-5-8243-1782-4.
Царенко А. П., Фёдоров Е. А.  Московский метрополитен имени В. И. Ленина: Справочник-путеводитель. — М.: Транспорт, 1984. — 224 с.
Юрков Д. Метро двойного назначения // Советские секретные бункеры (рус.). — 1. — Москва: Центр изучения современной фортификации и подземных сооружений, 2021. — 352 с. — ISBN 978-5-604604.

Ссылки

Официальный сайт Московского метрополитена
Сайты о Московском метро

Схема метрополитена с отображением подземных выходов и переходов
Московское метро (www.metro.ru) — авторский проект Артемия Лебедева, первый большой сайт о метро в рунете
Схема путевого развития Московского метрополитена
Тематические разделы о Московском метро

«Метрогипротранс. 70 лет — одна любовь, один проект» — проект Студии Артемия Лебедева
Московский метрополитен на UrbanRail.net (англ.)
Московский метрополитен на «Мире метро»
История Московского метрополитена в схемах на metroschemes.narod.ru
Интересные факты о московском метро на retromosfoto.ucoz.ru
Модель Московского метро в 3D
Форум Наш транспорт (ранее «Моё метро») — новости от работников и пассажиров метро
Тексты песен, посвящённых московскому метро или содержащих тему метро в тексте

## Title: `Вентиляция`

Вентиля́ция (от лат. ventilatio — проветривание) — перемещение газов под действием разности давления без применения замкнутых каналов.
Чаще всего применяется для удаления отработанного воздуха из помещения и замены его наружным. В необходимых случаях при этом проводится: кондиционирование воздуха, фильтрация, подогрев или охлаждение, увлажнение или осушение, ионизация и т. д. Вентиляция обеспечивает санитарно-гигиенические условия (температуру, относительную влажность, скорость движения воздуха и чистоту воздуха) воздушной среды в помещении, благоприятные для здоровья и самочувствия человека, отвечающие требованиям санитарных норм, технологических процессов, строительных конструкций зданий, технологий хранения и т. д.
Также под этим термином в технике часто имеются в виду системы оборудования, устройств и приборов для этих целей.

Исторический очерк
Отдельные приёмы организованной вентиляции закрытых помещений применялись ещё в древности. Вентиляция помещений до начала XIX века сводилась, как правило, к естественному проветриванию. Теорию естественного движения воздуха в каналах и трубах создал М. В. Ломоносов. В 1795 году В. X. Фрибе впервые изложил основные положения, определяющие интенсивность воздухообмена в отапливаемом помещении сквозь неплотности наружных ограждений, дверные проёмы и окна, положив этим начало учению о нейтральной зоне.
В начале XIX в. получает развитие вентиляция с тепловым побуждением приточного и удаляемого из помещения воздуха. Отечественные учёные отмечали несовершенство такого рода побуждения и связанные с ним большие расходы теплоты. Академик Э. X. Ленц указывал, что полная вентиляция может быть достигнута только механическим способом.
С появлением центробежных вентиляторов технология вентиляции помещений быстро совершенствуется. Первый успешно работавший центробежный вентилятор был предложен в 1832 году А. А. Саблуковым. В 1835 этот вентилятор был применён для проветривания Чагирского рудника на Алтае. Саблуков предложил его и для вентиляции помещений, трюмов кораблей, для ускорения сушки, испарения и т. д. Широкое распространение вентиляции с механическим побуждением движения воздуха началось с конца XIX века.
Одним из крупнейших учёных в области вентиляции и отопления являлся профессор В. М. Чаплин.
Одним из этапов развития вентиляции было появление электрических двигателей с изменяемой частотой оборотов. Первое упоминание о вентиляторе с таким электродвигателем ознаменовано 1972—1974 годами, когда компания Каналфлэкт применила этот двигатель в канальном вентиляторе.
Если же вести речь о вентиляции, как о явлении в истории, то нельзя не упомянуть Римскую империю, инженеры которой устанавливали в некоторых домах нечто вроде вентиляционной шахты. При этом термин «вентиляция» произошёл от латинского слова «ventilatio», которое означает «проветривание».

Вредные выделения в помещении
Основное назначение вентиляции — борьба с вредными выделениями в помещении. К вредным выделениям относятся:

избыточное тепло;
избыточная влага;
различные газы и пары вредных веществ, образующиеся как в бытовых (например, при приготовлении пищи или при использовании фумигаторов, свечей, при курении), так и в производственных процессах, таких как плавка и литьё металлов и сплавов, пластмасс, сварка и пайка, лакокрасочные работы, травление;
пыль.

Типы вентиляционных систем
Вентиляционная система — совокупность устройств для обработки, транспортирования, подачи и удаления воздуха.
Системы вентиляции классифицируются по следующим признакам:

По способу создания давления и перемещения воздуха: с естественным и искусственным (механическим) побуждением
По назначению: приточные и вытяжные
По способу организации воздухообмена: общеобменные, местные, аварийные, противодымные
По конструктивному исполнению: канальные и бесканальные
По количеству воздуха на человека в час. К примеру, в кухне при 4-конфорочной газовой плите 90 м3/ч, в совмещённом санузле 50 м3/ч, в бомбоубежище — не менее 2,5 м³/ч, в офисном помещении — не менее 20 м³ в час для посетителей, находящихся в помещении не более 2 часов, для постоянно находящихся людей — не менее 60 м³ в час.
Расчёт вентиляции производится с помощью следующих параметров: производительность по воздуху (м³/ч), рабочее давление (Па) и скорость потока воздуха в воздуховодах (м/с), допустимый уровень шума (дБ), мощность калорифера (кВт).
Норматив по воздухообмену регламентируется строительными нормами и правилами (СНиП) и санитарными нормами и правилами (Сан Пин)
Вентиляционная сеть.
Сетью называют систему воздуховодов и других элементов воздушного тракта, на которые подаёт воздух вентилятор. Сеть может состоять из элементов тракта, подсоединённых последовательно, параллельно или смешано.

Типы систем по способу побуждения движения воздуха
Естественная вентиляция
При естественной вентиляции воздухообмен осуществляется из-за разницы давления снаружи и внутри здания.
Под неорганизованной естественной системой вентиляции понимается воздухообмен в помещении, происходящий за счёт разности давлений внутреннего и наружного воздуха и действий ветра через неплотности ограждающих конструкций, а также при открывании форточек, фрамуг и дверей.
Организованной естественной вентиляцией называется воздухообмен, происходящий за счёт разности давлений внутреннего и наружного воздуха, но через специально устроенные приточные и вытяжные проёмы, степень открытия которых регулируется с помощью шиберов. Для создания пониженного давления в вентиляционном канале может использоваться дефлектор. Такой тип вентиляции чаще всего применяется в многоквартирных домах, однако у такой системы вентиляции есть существенный минус - возникновение обратной тяги, если температура в помещении ниже чем за его пределами, и сильное снижение или отсутствие тяги, если температура в помещении незначительно отличается от температуры воздуха за его пределами. В случае подключения, например, газоходов газовых котлов к такой системе вентиляции, это может привести к отравлению угарным газом.

Механическая вентиляция
При механической вентиляции воздухообмен происходит за счёт разности давления, создаваемой вентилятором или эжектором. Этот способ вентиляции более эффективен, так как воздух предварительно может быть очищен от пыли и доведён до требуемой температуры и влажности, к тому же, такая система ни при каких условиях не может создать обратную тягу. В механических системах вентиляции используются такие приборы и оборудование, как: вентиляторы, электродвигатели, фанкойлы, рекуператоры, регенераторы, воздухонагреватели, шумоглушители, пылеуловители и фильтры, осушители и увлажнители воздуха, ультрафиолетовые облучатели, частотные преобразователи, терморегуляторы, гигрометры, автоматика и др., позволяющие перемещать воздух в больших пространствах. Такие системы могут подавать и удалять воздух из локальных зон помещения в необходимом количестве, независимо от изменяющихся условий окружающей воздушной среды. При необходимости воздух подвергают различным видам обработки (очистке, нагреванию, увлажнению и т. д.), что практически невозможно в системах естественной вентиляции. Затраты электроэнергии на их работу могут быть довольно большими.
Примечание:
Следует отметить, что несчастные случаи могут происходить при одновременной работе газовых приборов (котлов, колонок, конвекторов) и вытяжного зонта над газовой плитой, работающего в режиме воздухоудаления. В результате работы «вытяжки» зачастую происходит опрокидывание тяги в дымовом канале и угарный газ вместе с продуктами сгорания от газового прибора поступает в помещение квартиры. Ситуация усугубляется, если в квартире установлены пластиковые окна. Их малая воздухопроницаемость приводит к недопустимому снижению количества приточного воздуха в квартиру (нарушается воздушный баланс). Проще говоря, установив новые окна, вы практически перекроете приток воздуха, необходимого как для полного сгорания газа, так и для нормальной работы общеобменной вентиляции.

Типы систем по назначению
Приточная вентиляция
Приточная система вентиляции — система, подающая в помещение определённое количество воздуха. В приборы в зависимости от бренда и модели могут быть встроены дополнительные функции:

очистка воздуха;
подогрев воздуха;
проветривание воздуха;
увлажнение воздуха;
рециркуляция воздуха.
На климатическом рынке есть вентиляционные установки, которые включают в себе часть функций и те, которые работают как универсальное устройство — сразу за несколько приборов.

Вытяжная вентиляция
Вытяжная вентиляция служит для удаления из помещения отработанного воздуха, а также продуктов сгорания природного газа от газовых плит.

Приточно-вытяжная вентиляция
Приточная-вытяжная система вентиляции — система, которая обеспечивает забор воздуха с улицы, его очистку от пыли, пыльцы и подачу в помещение. Одновременно вторая часть системы собирает отработанный воздух и неприятные запахи, и удаляет их наружу. Основным преимуществом приточно-вытяжной вентиляции является проветривание помещения при закрытых окнах без шума, пыли, сквозняков и аллергенов.

Типы систем по способу организации воздухообмена
Общеобменная вентиляция
Общеобменная система вентиляции предусматривается для создания одинаковых условий и параметров воздушной среды (температуры, влажности и подвижности воздуха) во всём объёме помещения, главным образом в его рабочей зоне (1,5—2,0 м от пола), когда вредные вещества распространяются по всему объёму помещения и нет возможности (или нет необходимости) их уловить в месте образования.

Местная вентиляция
Местной вентиляцией называется такая, при которой воздух подают на определённые места (местная приточная вентиляция) и загрязнённый воздух удаляют только от мест образования вредных выделений (местная вытяжная вентиляция).
Местная приточная вентиляция может обеспечивать приток чистого воздуха (предварительно очищенного и подогретого) к определённым местам. И наоборот, местная вытяжная вентиляция удаляет воздух от определённых мест с наибольшей концентрацией вредных примесей в воздухе. Примером такой местной вытяжной вентиляции может быть вытяжка на кухне, которая устанавливается над газовой или электрической плитой. Чаще всего используются такие системы в промышленности.

Аварийная вентиляция
Аварийная система вентиляции устанавливается в помещениях, где возможен неожиданный выброс чрезвычайно опасных вредных веществ в количествах, значительно превышающих ПДК, с целью их быстрого удаления. Аварийная вентиляция необходима для удаления газа в помещениях с газовым пожаротушением, для удаления газа после работы системы.

Противодымная вентиляция
Противодымная система вентиляции устанавливается в производственных зданиях, где применяются технологии с повышенной пожароопасностью, и служит для обеспечения эвакуации людей. С помощью этой системы подаётся необходимое количество воздуха, препятствующего распространению дыма в помещении. Система работает в начальной стадии пожара.

Типы систем по конструктивному исполнению
Канальная вентиляция
Канальные системы вентиляции имеют сеть воздуховодов для перемещения воздуха.

Бесканальная вентиляция
При бесканальной системе вентилятор устанавливают в стене, перекрытии.

Вентиляционное оборудование
Системы вентиляции включают в себя группы самого разнообразного оборудования: прежде всего, это вентиляторы, вентиляторные агрегаты или вентиляционные установки. Среди дополнительного оборудования — шумоглушители, воздушные фильтры, электрические и водяные воздухонагреватели, регулирующие и воздухораспределительные устройства и пр.

Вентиляторы
Вентилятор представляет собой механическое устройство, предназначенное для перемещения воздуха по воздуховодам системы вентиляции. По конструкции и принципу действия вентиляторы делятся на канальные (круглые и прямоугольные), крышные, осевые (аксиальные), центробежные (радиальные) и тангенциальные (диаметральные), батутные и т. д.

Осевые вентиляторы
Осевой вентилятор представляет собой расположенное в цилиндрическом кожухе (обечайке) колесо из консольных лопастей, закреплённых на втулке под углом к плоскости вращения. Рабочее колесо как правило насаживается непосредственно на ось электродвигателя.
При вращении колеса воздух захватывается лопастями и перемещается в осевом направлении. При этом перемещение воздуха в радиальном направлении практически отсутствует.
Осевые вентиляторы имеют больший КПД по сравнению с радиальными и диаметральными. Такие вентиляторы, как правило, применяют для подачи значительных объёмов воздуха при малых аэродинамических сопротивлениях вентиляционной сети.

Центробежные (радиальные) вентиляторы
Центробежный (радиальный) вентилятор представляет собой расположенное в спиральном кожухе лопаточное (рабочее) колесо, при вращении которого воздух, попадающий в каналы между его лопатками, двигается в радиальном направлении к периферии колеса и сжимается. Под действием центробежной силы он отбрасывается в спиральный кожух и далее направляется в нагнетательное отверстие.
В зависимости от назначения вентилятора, лопатки рабочего колеса изготавливают загнутыми вперёд или назад. Количество лопаток бывает различным в зависимости от типа и назначения вентилятора. Применение радиальных вентиляторов с лопатками, загнутыми назад, даёт экономию электроэнергии примерно 20 %. Также они легко переносят перегрузки по расходу воздуха. Преимуществами радиальных вентиляторов с лопатками рабочего колеса, загнутыми вперёд, являются меньший диаметр колеса, а соответственно и меньшие размеры самого вентилятора, и более низкая частота вращения, что создаёт меньший шум.

Диаметральные (тангенциальные) вентиляторы
Диаметральный (тангенциальный) вентилятор состоит из рабочего колеса барабанного типа с загнутыми вперёд лопатками и корпуса, имеющего патрубок на входе и диффузор на выходе. Действие диаметральных вентиляторов основано на двукратном поперечном прохождении потока воздуха через рабочее колесо.
Используются в основном в кондиционерах (внутренние блоки сплит-систем) и тепловых завесах. В вентиляционных сетях диаметральные вентиляторы используются крайне редко.

Шумоглушители
Установка в систему вентиляции шумоглушителей является одной из эффективных мер по снижению аэродинамического шума в воздушном потоке. Наиболее часто применяемые шумоглушители конструктивно делятся на пластинчатые и трубчатые. Главная их особенность — наличие развитых поверхностей, облицованных звукопоглощающим материалом (минеральная вата, стекловолокно и прочее).
Чаще всего шумоглушитель устанавливается на определённом расстоянии между вентилятором и магистральным воздуховодом на границе естественной преграды шума (любая звуковая преграда).
Необходимость установки шумоглушителя в вентиляционной системе должна быть подтверждена специальным акустическим расчётом.

Воздушные фильтры
Служат для очистки приточного воздуха, а в некоторых случаях и вытяжного воздуха.
Существует множество типов конструкций воздушных фильтров. Принцип действия, конструкция и материал фильтра зависят от требуемых параметров воздуха.
В вентиляционных системах воздушные фильтры классифицируются по степени очистки воздуха. Чем меньше частички пыли, эффективно улавливаемые фильтром, тем выше его класс очистки. Согласно принятой международной классификации, существует четыре класса фильтров грубой очистки воздуха (классы G1-G4), пять классов тонкой очистки (классы F5-F9), пять классов фильтров особо тонкой очистки, именуемых так же HEPA-фильтрами (классы H10-H14), а также три класса ультра-тонкой очистки воздуха, или ULPA-фильтры (классы U15-U17).
Помимо класса очистки, важными параметрами фильтров являются их пылеёмкость и аэродинамическое сопротивление.

Воздухонагреватели, канальный нагреватель (или калори́фер)
В современных зданиях система вентиляции, как правило, работает совместно с системой отопления здания, а в некоторых случаях полностью её заменяет. Для подогрева воздуха в вентиляционных системах используются воздухонагреватели. Большинство воздухонагревателей в вентиляционных системах — водяные либо электрические.
Водяные воздухонагреватели это по сути теплообменники, в которых воздух получает тепло от горячей воды, нагретой в отопительном котле или поступающей из центральной теплосети.
Электрические воздухонагреватели питаются от электросети и преобразуют электрическую энергию в тепловую.
Кроме активных воздухонагревателей применяются пассивные системы рекуперации тепловой энергии. Рекуперация осуществляется за счёт теплообмена между вытяжным каналом и притоком. Теплообмен может производиться при помощи нескольких основных типов теплообменников:

перекрестноточный пластинчатый рекуператор, в котором несмешивающиеся воздушные потоки притока и вытяжки текут по многочисленным каналам с общими для разных потоков стенками — рекуперация тепла по разным данным может составлять от 70 до 85 %;
роторный рекуператор, в котором теплообмен происходит в роторе, при этом происходит частичное смешение приточного и вытягиваемого воздуха — степень рекуперации тепла аналогична показателям пластинчатого рекуператора;
рекуператор с промежуточным теплообменником, в котором линии притока и вытяжки разнесены в пространстве на некоторое расстояние, а перенос тепла между вентканалами осуществляется путём перекачки жидкого теплоносителя между индивидуальными теплообменниками в каналах — степень рекуперации тепла в таких системах достигает 50-60 %, но бывает оправдана, когда необходимо гарантированно исключить воздухообмен между вытяжкой и притоком.

Воздуховоды
Противопожарные и регулирующие клапаны
Клапан противопожарный — автоматически и дистанционно управляемое устройство для перекрытия вентиляционных каналов или проёмов в ограждающих строительных конструкциях зданий, имеющее предельные состояния по огнестойкости, характеризуемые потерей плотности и потерей теплоизолирующей способности:

нормально открытый (закрываемый при пожаре);
нормально закрытый (открываемый при пожаре);
двойного действия (закрываемый при пожаре и открываемый после пожара).

Регулирующий клапан предназначен для использования в системах отопления, кондиционирования, вентиляции и служит для регулирования, перекрытия или изменения направления потока воздуха в вентиляционных каналах.
Корпус клапана — неподвижный элемент конструкции клапана, который устанавливается в монтажном проёме ограждающей конструкции или на ответвлении воздуховода.
Заслонка клапана — подвижный элемент конструкции клапана, установленный в корпусе и перекрывающий его проходное сечение.
Привод клапана — механизм, обеспечивающий перевод заслонки в автоматическом и дистанционном режимах в положение, соответствующее его функциональному назначению. Привод клапана является исполнительным механизмом.
Одной из главных характеристик клапана является тип привода клапана.

Пружинный привод с тепловым замком
Пружинный привод с тепловым замком дешевле остальных и не требует дополнительной автоматики и подвода электропитания. Однако он имеет ряд существенных недостатков:

срабатывание привода происходит только после расплавления теплового замка, для этого необходимо, чтобы горячие продукты горения достаточно длительное время проходили через клапан и омывали тепловой замок. Привод в результате этого имеет большую инерцию и срабатывает не в начале пожара, а значительно позже;
невозможно включение привода от внешнего устройства. Это не позволяет периодически проверять работоспособность клапана и включать его в случае пожара вручную;
после срабатывания требуется замена клапана или его теплового замка, в результате после однократного срабатывания система оказывается незащищённой.

Пружинный привод с электромагнитной защёлкой
Данный привод также называют электромагнитным.
В 1950 году в СССР началось внедрение на мукомольных производствах рециркуляционных установок, имеющих в своём составе клапан с электромагнитной защёлкой, который при пропадании напряжения закрывался под собственным весом и весом груза. При возникновении пожара в вентиляционный сети срабатывал тепловой извещатель, установленный внутри воздуховода. Происходило закрытие клапана и отключение вентилятора.
В клапанах современных конструкций срабатывание происходит при подаче напряжения на электромагнит. Защёлка при срабатывании освобождает заслонку и под действием пружины клапан переходит в рабочее положение. Возврат заслонки в исходное положение после срабатывания клапана производится вручную.

Электромеханический (электромоторный) привод с возвратной пружиной
Электромеханический привод часто называют по названию швейцарской фирмы BELIMO.
Управляющим сигналом на срабатывание клапанов является снятие напряжения с привода, после чего возвратная пружина переводит заслонку в рабочее положение. Подача напряжения на привод электродвигателя переводит заслонку в исходное положение и удерживает её, потребляя при этом незначительную мощность.

Электромеханический (электромоторный) реверсивный привод
При смене полярности напряжения на электромоторе происходит изменение положения заслонки на противоположное. Применяется в основном в автотехнике, например заслонки системы климат-контроля. Часто установлены концевые выключатели для отключения питания электромотора в конечном положении заслонки. Установить промежуточное состояние привода невозможно: только полностью открыт или закрыт.
Электромеханический (электромоторный) привод с цифровым (микропроцессорным) управлением
Применяется в «умных» системах вентиляции зданий. В основном управление осуществляется постоянным напряжением 0…10 Вольт постоянного тока. Данный тип привода позволяет открывать заслонку на необходимую величину и тем самым регулировать поток воздуха и пропускную способность. Часто самодельные устройства подобного типа используются в самодельных инкубаторах, рекуператорах и иных системах.

См. также
Аспирация (вентиляция)
Вентилятор
Дымоудаление
Воздухообрабатывающая установка

Примечания
Литература
Капустин М. Я., Нюберг А. Г., Тимонов В. Е. Вентиляция зданий // Энциклопедический словарь Брокгауза и Ефрона : в 86 т. (82 т. и 4 доп.). — СПб., 1890—1907.
Вентиляция // Краткая энциклопедия домашнего хозяйства. — М.: Государственное Научное издательство «Большая Советская энциклопедия», 1959.

Ссылки

Савельев Н. Ф. Электрическая вентиляция // Энциклопедический словарь Брокгауза и Ефрона : в 86 т. (82 т. и 4 доп.). — СПб., 1890—1907.
Вентиляция // Большая советская энциклопедия : [в 30 т.] / гл. ред.  А. М. Прохоров. — 3-е изд. — М. : Советская энциклопедия, 1969—1978.

Загружено 5 статей


In [33]:
for i in range(5):
    print(f'title_idx: {i}, text_len: {len(corpus[i]['text'])}')


title_idx: 0, text_len: 43136
title_idx: 1, text_len: 15016
title_idx: 2, text_len: 9266
title_idx: 3, text_len: 128215
title_idx: 4, text_len: 22340


In [8]:
# Единый интерфейс для двух вариантов векторизации: dense embeddings и fallback.
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfFallbackBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray()
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray()
        return vectors.astype("float32")


def build_embedding_backend(
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
) -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(model_name=model_name, device=device)
        print("Используем полноценные dense embeddings.")
        print("Бэкэнд:", backend.backend_name)
        return backend
    except Exception as e:
        print("Не удалось загрузить sentence-transformers encoder.")
        print("Причина:", repr(e))
        print("Переключаемся на TF-IDF fallback. Ноутбук останется рабочим,")
        print("но это уже не полноценные dense embeddings.")
        return TfidfFallbackBackend()


embedder = build_embedding_backend(device=DEVICE)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13191.51it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
